# Data Analysis & Visualization Notebook

## Overview

Brief description of your analysis objectives and the data sources
you’ll be working with.



## Setup & Dependencies


In [1]:
import os, sys, json, datetime, re, math  # Provides OS-dependent functionality, system-specific parameters, JSON handling, and date/time manipulation
import pandas as pd             # Provides data structures and data analysis tools
import numpy as np              # Supports large, multi-dimensional arrays and matrices
import requests
import time
from tqdm import tqdm
import glob as glob

#thi data contants
from cprl_functions.defined_functions import *
from cprl_functions.state_capture import *
from cprl_functions.text_printing import bordered
from cprl_functions.data_packet_defs import *

#Import Data
###################
import data_collection.pull_data as data_pull
from graphs.viz_graphs_template import *


# Create folder if needed
graphs_dir = r"C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data Packets\K-12\graphs"

import matplotlib.pyplot as plt
from matplotlib import font_manager

font_dirs = [r"C:\Users\clutz\project_folder\Projects\fonts"]  # The path to the custom font file.
font_files = font_manager.findSystemFonts(fontpaths=font_dirs)

import os

font_dir = r"C:\Users\clutz\project_folder\Projects\fonts"
# print([f for f in os.listdir(font_dir) if f.lower().endswith(".ttf")])


for font_file in font_files:
    font_manager.fontManager.addfont(font_file)

print(state_abbreviations_priority)



['AZ', 'CA', 'CT', 'DC', 'DE', 'FL', 'GA', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'MA', 'MD', 'MI', 'MN', 'MO', 'NC', 'ND', 'NE', 'NJ', 'NM', 'NY', 'OH', 'OK', 'OR', 'RI', 'SC', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']


## Fonts

In [ ]:
from matplotlib import font_manager
import matplotlib as mpl
import os

font_dir = r"C:\Users\clutz\project_folder\Projects\fonts"

# 1. Verify TTF files exist
print("Fonts found in folder:")
print([f for f in os.listdir(font_dir) if f.lower().endswith(".ttf")])

# 2. Register custom fonts
font_files = font_manager.findSystemFonts(fontpaths=[font_dir])
for font_file in font_files:
    font_manager.fontManager.addfont(font_file)

# 3. Rebuild font cache properly
font_manager._load_fontmanager(try_read_cache=False)

# 4. Check for Lato fonts
lato_fonts = [f.name for f in font_manager.fontManager.ttflist if "Lato" in f.name]
print("Lato fonts recognized:", lato_fonts)


In [ ]:
import plotly.io as pio

pio.templates.default = None
pio.templates["custom"] = pio.templates["plotly"]

pio.templates["custom"].layout.font = {
    "family": "Lato", 
    "color": hunt_darkgray
}


# Graph Controls

In [2]:
#graph widget
import json, os
import ipywidgets as widgets
from IPython.display import display
from types import SimpleNamespace

# -----------------------------
# Config
# -----------------------------
SETTINGS_FILE = "toggle_settings.json"
DEFAULTS = {"save_it": False, "show_it": True, "just_one": False, "subset_states": None}


# -----------------------------
# Helpers
# -----------------------------
def load_settings(defaults=None):
    data = {}
    if os.path.exists(SETTINGS_FILE):
        try:
            with open(SETTINGS_FILE, "r") as f:
                content = f.read().strip()
                if content:
                    data = json.loads(content)
        except json.JSONDecodeError:
            print("Warning: settings file corrupted or empty. Recreating with defaults.")
            data = {}
    if defaults:
        for k, v in defaults.items():
            data.setdefault(k, v)
    return data


def save_settings(data):
    """Write settings to disk immediately."""
    with open(SETTINGS_FILE, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Saved settings → {SETTINGS_FILE}")


def dict_to_namespace(d):
    return SimpleNamespace(**d)


# -----------------------------
# Load saved or default settings
# -----------------------------
settings_data = load_settings(DEFAULTS)
settings = dict_to_namespace(settings_data)


# -----------------------------
# Create widgets
# -----------------------------
save_toggle = widgets.Checkbox(value=settings.save_it, description="Save graphs")
show_toggle = widgets.Checkbox(value=settings.show_it, description="Show graphs")
just_one_toggle = widgets.Checkbox(value=settings.just_one, description="Just one graph")
# subsetting_states = widgets.Dropdown(options = state_abbreviations_priority+['None'],value='None',description='State',disabled=False)
subsetting_states = widgets.SelectMultiple(
    options=state_abbreviations_priority+['None'],
    value=['None'],
    #rows=10,
    description='States',
    disabled=False
)
# -----------------------------
# Unified event handler
# -----------------------------
def on_toggle_change(change):
    # Update dict + namespace
    settings_data["save_it"] = save_toggle.value
    settings_data["show_it"] = show_toggle.value
    settings_data["just_one"] = just_one_toggle.value
    settings_data["subset_states"] = subsetting_states.value

    settings.save_it = save_toggle.value
    settings.show_it = show_toggle.value
    settings.just_one = just_one_toggle.value
    settings.subset_states = subsetting_states.value

    save_settings(settings_data)  # persist immediately


# Attach handler to all three widgets
for toggle in (save_toggle, show_toggle, just_one_toggle, subsetting_states):
    toggle.observe(on_toggle_change, names="value")

# -----------------------------
# Display
# -----------------------------
display(widgets.VBox([save_toggle, show_toggle, just_one_toggle, subsetting_states]))


In [3]:
sizing = {
    '1.1':(210,155),
    '1.2':(505,91),#these two should be the same sime to keep consistency, 
    '1.3':(505,91),
    '1.4':(505,91),
    '2':(252,200),
    '3':(252,144),
    '4':(133,112),
    '4.5':(252,108),
    '4.6':(252,108),
    '5':(260,112),
    '5.3':(235,95),
    '5.7':(245,95),
    '6.1':(490,120),
    '6.2':(260,100),
    '6.3':(485,110),
    '7.1':(504,130),
    '7.2':(504,130),
    '7.3':(504,130),
    '8.1':(505,95),
    '8.2':(540,145),
    '8.3':(165,94),
    '8.4':(165,94),
    '8.5':(165,94),
    '9.1':(510,120),
}

def get_res(val, res): 
    val = val*res
    return val 
    
def get_sizing(graph_num, res=5):

    five_match=re.search(r'^5.\d', str(graph_num))
    four_match = re.search(r'^4.(\d)', str(graph_num))
    if four_match is not None:
        if int(four_match.group(1))<5:
            output = sizing.get('4')
        elif int(four_match.group(1))>=5:
            output = sizing.get(graph_num)

    elif five_match is not None:
        output = sizing.get('5')
    else:
        output = sizing.get(graph_num)
    # print(f'output raw: {output}')
    


    output = tuple([get_res(x, res) for x in output])
    # print(output)


    return output


get_sizing('8.3', 5)

(825, 470)

In [ ]:
data_pull.get_ap_graph_data()

In [ ]:
# state_abbreviations_priority

## Helper Functions

In [4]:
def get_val_offset(primary,secondary, offset):
    import math
    if '‡' in str(primary) or '‡' in str(secondary) or math.isnan(primary) or math.isnan(secondary):
        return np.nan
    
    try:
        if primary > secondary:
            return primary + offset
        elif primary < secondary:
            return primary - offset
    except:
        print(primary)
        print(type(primary))
        print(secondary)
        print(type(secondary))
    

In [5]:
#save to folder
graphs_dir = r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs'
# graphs_dir = r'C:\Users\clutz\Downloads\k12 datapackets'

def save_to_folder(fig,filename, g_width,g_height, state):
    # Create folder if needed

    output_folder = os.path.join(graphs_dir,state)
    os.makedirs(output_folder, exist_ok=True)

    # Save the file
    fig.write_image(
        
        os.path.join(output_folder,filename),
        width=g_width,
        height=g_height,
        scale=1  # For higher resolution (2x default)
    )


In [6]:
def position_calc(val1, val2, offset=7, biggap=0, superthreshold=2, threshold=18, multiplier=1.5):
    """
    Calculate label position based on distance between two values.
    
    Args:
        val1: Value to position
        val2: Reference value to compare against
        offset: Base offset distance
        biggap: Additional gap for widely separated values
        superthreshold: Distance below which values are "super close"
        threshold: Distance above which values are "far apart"
        multiplier: Scaling factor for super close adjustments
    """
    # Handle NaN values
    if pd.isna(val1) or pd.isna(val2):
        return None
    
    distance = val1 - val2
    
    # Determine direction
    if distance == 0:
        # If values are identical, offset upward slightly
        return val1 + offset
    
    is_above = distance > 0
    abs_distance = abs(distance)
    
    # Categorize proximity
    is_super_close = abs_distance < superthreshold
    is_close = abs_distance <= threshold
    
    # Calculate adjustment
    if is_super_close:
        # Very close: push away more aggressively
        adjustment = offset * multiplier
    elif is_close:
        # Moderately close: standard offset
        adjustment = offset - 1
    else:
        # Far apart: minimal offset
        adjustment = offset - biggap
        
    
    # Apply direction
    if is_above:
        new_position = val1 + adjustment
    else:
        new_position = val1 - adjustment
    
    return new_position

In [7]:
# def position_calc(val1, val2, offset=6, biggap = 0, superthreshold = 4,threshold = 18, multiplier = 1.75):
#     distance_away = (val1-val2)
#     if isinstance(val1, float) and 'nan' in str(val1).lower():
#         return None
#     charge = None
#     if distance_away <0:
#         charge = 'neg'
#     elif distance_away >0:
#         charge = 'pos'
    
    
#     close_up = True
#     super_close = False
#     if abs(distance_away)>threshold:
#         close_up = False
#     if abs(distance_away)<superthreshold:
#         super_close = True
    
#     # print(f'intitial distance: {distance_away}')
#     # print(f'charge: {charge}')
#     # print(f'not_close: {close_up}')
#     # print(f'super_close: {super_close}')

    
#     if charge == 'pos':
#         if close_up == True:
#             distance_away = (offset - 1)
#         elif close_up == False:
#             if super_close == True:
#                 distance_away += offset*multiplier
#             if super_close == False:
#                 distance_away = offset-biggap
#     elif charge == 'neg':

#         if close_up == True:
#             distance_away = -(offset - 1)
#         elif close_up == False:
#             if super_close == True:
#                 distance_away -= offset*multiplier
#             if super_close == False:
#                 distance_away = -(offset-biggap)



#     # print(distance_away)
#     new_val = val1+(distance_away)

#     # if new_val<17:
#     return new_val

#     # pos = None
#     # if val1>val2:
#     #     pos = 'top center'
#     # else:
#     #     pos = 'bottom center'
#     # return pos

In [8]:
def get_highlight_color(x,state):
    if x==state:
        return hunt_dark_purple
    else:
        return hunt_light_purple
    

In [ ]:
# data_directory = Path(r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data Packets\k12\data')
# def get_files():
#     file_dict = {}
#     for file in data_directory.iterdir():
#         print(file.name)
#         if 'naep' in file.name.lower():
#             naep_file = data_directory / file.name
#             file_dict['naep'] = naep_file
#         elif 'data_collection' in file.name.lower():
#             data_collection_file = data_directory / file.name
#             file_dict['data_collection'] = data_collection_file
        

#     return file_dict


        



# with open(data_directory / naep_file, 'r') as file:
#     excel_file = pd.ExcelFile(file,engine='calamine')
#     for sheet in excel_file.sheet_names:
#         if '- 2024' in sheet:
#             print(sheet) 

# excel_file = pd.ExcelFile(data_directory / data_collection_file, engine='calamine')
# for sheet in excel_file.sheet_names:
#     print(sheet)
#     if 'merge' in sheet.lower():
#         merge_tags = excel_file.parse(sheet_name=sheet)
#         break

# print(merge_tags.to_string())



# glob_pat = os.path.join(filepath, 'NAEP*.xlsx')
# files = glob.glob(glob_pat)
# print(merge_tags)
# for file in filepath:
# pd.ExcelFile()

In [9]:
def edit_year(year_val, delim):
    years_split = year_val.split(delim,1)
    new_years = []
    for year in years_split:
        if len(year)==4:
            new_years.append(int(year))
        else:
            year_alt = f'20{year}'
            new_years.append(int(year_alt))
    new_years = sorted(new_years)
    year_reformat = f'{new_years[0]}-{"<br>"}{new_years[1]}'
    return year_reformat

# Merge File

In [17]:
merge_dir = Path(r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\merge_files')
merge = True

merge_dfs = {}
for root, dir, files in os.walk(merge_dir):
    if len(files)!=0:
        # print(root)
        file = files[0]
        if len(file.replace('merge.csv','').strip()) == 0:
            continue
        print(files)
        print(file)
        df = pd.read_csv(os.path.join(root,file)).reset_index(drop=True)
        if any('Unnamed: 0' in str(x) for x in df.columns):
            df = df.drop(columns = ['Unnamed: 0'])
        

        # print(df.to_string())
        file_type =root.split('\\')[-1]
        # print(type)
        merge_dfs[file_type] = df
print(merge_dfs.keys())


['merge_bavery.csv', 'merge_jcraig.csv', 'merge_JOrtiz.csv']
merge_bavery.csv
['graphs_merge_file.csv']
graphs_merge_file.csv
['merge_text_file.csv']
merge_text_file.csv
dict_keys(['merge_files', 'graphs', 'text'])


In [18]:
# get merge file

if merge == True:
    

    #merge all data
    merge_file = merge_dfs.get('text').merge(merge_dfs.get('graphs'), on='state_abrv')

    # get_user for filename
    links = get_col_uniq_vals(merge_file['@num_trad'])
    user = list(set([x.split(':')[2] for x in links]))[0]
    # print(link)
    
    merge_file = merge_file[merge_file['state_abrv'].isin(settings.subset_states)].reset_index(drop = True)
    print(merge_file)
    merge_file.to_csv(os.path.join(merge_dir,f'merge_{user}.csv'), index=False)


  state_abrv   00 State Number of Public Schools  \
0         FL    Florida                    4,229   
1         UT       Utah                    1,096   
2         WI  Wisconsin                    2,224   

   Number of Public School Districts Per-Pupil Expenditure  \
0                                 82            $11,075.89   
1                                155             $9,551.93   
2                                468            $14,504.94   

   State Rank - Per-Pupil Expenditure  Student-Teacher Ratio  \
0                                   6                   18.4   
1                                   1                   20.6   
2                                  26                   13.7   

  Years - Student-Teacher Ratio  State Rank - Student-Teacher Ratio  \
0                     2023-2024                                  48   
1                     2023-2024                                  49   
2                     2023-2024                                  21   



# Graphs

## Page 1

In [12]:
# pull in enrollment data
enroll_total_df = data_pull.total_enroll_data_clean()
enroll_re_df = data_pull.re_enroll_data_clean(enroll_total_df)

print(enroll_total_df.to_string())
# print(enroll_re_df.to_string())
settings.subset_state

   state_abrv     year total_enrollment  fy
1          AL  2023-24           748650  24
2          AK  2023-24           131243  24
3          AZ  2024-25          1099612  25
4          AR  2024-25           474337  25
5          CA  2024-25          5806221  25
6          CO  2024-25           881065  25
7          CT  2024-25           508402  25
8          DE  2024-25           142495  25
9          FL  2024-25          2859655  25
10         GA  2024-25          1736730  25
11         HI  2024-25           165340  25
12         ID  2024-25           306937  25
13         IL  2024-25          1851290  25
14         IN  2024-25          1040190  25
15         IA  2024-25           475459  25
16         KS  2024-25           476833  25
17         KY  2023-24           657520  24
18         LA  2023-24           708190  24
19         ME  2023-24           172545  24
20         MD  2024-25           891553  25
21         MA  2024-25           915932  25
22         MI  2024-25          

'CA'

### 1.1 Total K-12 Enrollment

In [13]:
#creates table 1.1
from graphs.viz_graphs_template import *
import plotly.graph_objects as go
import plotly.express as px

g_width,g_height = get_sizing('1.1')
axis_font_size = 35

label_font_size = 28
# g_width = 110
# g_height = 915
offset_control = .10
multiplier = 1.1
filename = f'1.1.png'
enroll_total_df = data_pull.total_enroll_data_clean()

for jur in state_abbreviations_priority:
    if settings.subset_states != 'None' and jur not in settings.subset_states:
        continue

    if jur == 'US':
        continue
    print(jur)
    #filter for only state vals
    result = enroll_total_df[enroll_total_df['state_abrv']==jur]
    print(result.to_string())
    years = [edit_year(x,'-') for x in result['year'].to_list()]
    # print(years)
    #call graph
    fig = graph_1_1(years, result['total_enrollment'], g_width, g_height)
    # print(type(fig))
    
    #position calculations for text annotations
    ymin = min(result['total_enrollment'])
    ymax = max(result['total_enrollment'])
    yrange = ymax - ymin
    offset = yrange * offset_control  # 5% of the total vertical range
    range_offset = yrange * (offset_control*2)

    enrollment_text = result['total_enrollment'].to_list()
    text_y_positions = []  # Store y-positions for text    
    alignment = []
    for i, x in enumerate(enrollment_text):
        if i == 0:
            # First element - compare with next
            next_val = enrollment_text[i+1]
            if next_val > x:
                value = x - offset  # Position below
                alignment.append('top right')
            else:
                value = x + offset  # Position above
                alignment.append('middle center')
                
        elif i < len(enrollment_text) - 1:  # Fixed: was missing "- 1"
            # Middle elements - has both previous and next
            next_val = enrollment_text[i+1]
            last = enrollment_text[i-1]
            
            # Calculate slopes (direction of change)
            slope_to = x - last  # Positive if increasing
            slope_from = next_val - x  # Positive if will increase
            
            if slope_to > 0 and slope_from > 0:
                # Going up and will continue up
                value = x + (multiplier * offset)
                alignment.append('top left')
            elif slope_to < 0 and slope_from < 0:
                # Going down and will continue down
                value = x + (offset*multiplier)
                alignment.append('middle right')
            elif slope_to > 0 and slope_from < 0:
                # Peak (was going up, now going down)
                value = x + offset
                alignment.append('top center')
            elif slope_to < 0 and slope_from > 0:
                # Valley (was going down, now going up)
                value = x - offset
                alignment.append('bottom center')
            else:
                # Flat
                value = x + offset
                alignment.append('top center')

                
        else:
            # Last element - compare with previous
            last = enrollment_text[i-1]
                
            if x > last:
                if last==ymin:
                    value = x + (offset/2)  # Was increasing, put above
                    alignment.append('middle left')
                else:
                    value = x + offset  # Was increasing, put above
                    alignment.append('top left')

            else:
                if last==ymin:
                    value = x - (offset/2)  # Was increasing, put above
                    alignment.append('middle left')
                else:
                    value = x - offset  # Was decreasing, put below
                    alignment.append('middle left')

        text_y_positions.append(value)
    print(len(alignment))
    print(len(text_y_positions))
    # Add text labels
    fig.add_trace(go.Scatter(
        x=years,
        y=text_y_positions,  # Use calculated positions
        mode='text',
        text=[f"{int(val):,}" for val in enrollment_text],  # Format numbers with commas
        textposition=alignment,
        textfont=get_base_text(label_font_size, bold = True),
        showlegend=False
    ))
    
    fig.update_yaxes(
        showticklabels=False,  # Hide tick labels
        showgrid=False,        # Hide gridlines
        zeroline=False,         # Hide zero line
        showline = False,
        range=[ymin - range_offset, ymax + range_offset]
    )   
    fig.update_xaxes(
        tickfont = get_base_text(axis_font_size),
        tickangle = 0
    ) 
    fig.update_layout(
        margin = dict(l=10, r=10, t=10, b=10)
    ) 
    
    if settings.show_it:
        fig.show()
        if settings.just_one:
            break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,jur)

    
    # break

   state_abrv     year total_enrollment  fy
1          AL  2023-24           748650  24
2          AK  2023-24           131243  24
3          AZ  2024-25          1099612  25
4          AR  2024-25           474337  25
5          CA  2024-25          5806221  25
6          CO  2024-25           881065  25
7          CT  2024-25           508402  25
8          DE  2024-25           142495  25
9          FL  2024-25          2859655  25
10         GA  2024-25          1736730  25
11         HI  2024-25           165340  25
12         ID  2024-25           306937  25
13         IL  2024-25          1851290  25
14         IN  2024-25          1040190  25
15         IA  2024-25           475459  25
16         KS  2024-25           476833  25
17         KY  2023-24           657520  24
18         LA  2023-24           708190  24
19         ME  2023-24           172545  24
20         MD  2024-25           891553  25
21         MA  2024-25           915932  25
22         MI  2024-25          

UT
    state_abrv  fy     year total_enrollment
308         UT  18  2017-18         668274.0
309         UT  19  2018-19         677031.0
310         UT  20  2019-20         684694.0
311         UT  21  2020-21         680659.0
312         UT  22  2021-22         690934.0
313         UT  23  2022-23         691906.0
314         UT  24  2023-24         689883.0
398         UT  25  2024-25           667789
8
8


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




WI
    state_abrv  fy     year total_enrollment
343         WI  18  2017-18         860753.0
344         WI  19  2018-19         859333.0
345         WI  20  2019-20         855400.0
346         WI  21  2020-21         830066.0
347         WI  22  2021-22         829359.0
348         WI  23  2022-23         823040.0
349         WI  24  2023-24         814202.0
402         WI  25  2024-25           805881
8
8


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 1.2 Enrollment by Race and Ethnicity 

In [14]:
#creates GRAPH 1.2 (example of 100% bar)
from graphs.viz_graphs_template import *
import plotly.graph_objects as go
import plotly.express as px



#graph settings

aspect_ratio = 7
g_width,g_height = get_sizing('1.2', )
axis_font_size = 45
# graph_h = 458
# graph_wd = 2705


multiplier = 1.5
filename = f'1.2.png'
enroll_re_df = data_pull.re_enroll_data_clean(enroll_total_df)


us_values = enroll_re_df[enroll_re_df['state_abrv']=="US"]

us_values = us_values.sort_values('year', ascending=False).reset_index(drop=True)
# print(us_values[us_values['fy']==us_values['fy'].max()].to_string())
# us_total
for jur in state_abbreviations_priority:
    if settings.subset_states != 'None' and jur not in settings.subset_states:
        continue
    if jur == 'US':
        continue
    # print(jur)
    
    #fetch for state
    result = enroll_re_df[enroll_re_df['state_abrv']==jur]
    print(result.to_string())
    
    us_series = us_values[us_values['fy']==us_values['fy'].max()]
    
    #set up/cleaning
    series_dict = {'state':result,'us':us_series}
    
    
    if all(len(x)==1 for x in series_dict.values()):
        # print('good to keep going')
        print(series_dict.get('state').columns)
        categories = series_dict.get('state').columns[5:]
        state_vals = series_dict.get('state').iloc[0,5:].to_list()
        us_vals = series_dict.get('us').iloc[0,5:].to_list()
        both_values_dict = {"us":us_vals, 'state':state_vals}
        for k,v in both_values_dict.items():
            print(k)
            print(v)
            new_values = [float(x) for x in v]
            both_values_dict[k] = new_values

        
        # print(state_percents)
        # us_values = series_dict.get('state').columns[5:]
        categories = [x.replace('_'," ").title() for x in categories]
        print(f'categories: {categories}')
        print(f'state vals: {both_values_dict.get('state')}')
        print(f'us vals: {both_values_dict.get('us')}')
        
        
        fig = graph_1_2(categories, both_values_dict.get('state'), both_values_dict.get('us'), jur, g_width, g_height)
        fig.update_yaxes(tickfont=get_base_text(axis_font_size))
        fig.update_layout(margin=dict(l=200, r=20, t=100, b=100))
        # save_to_folder(fig,filename,g_width,g_height,state)
        if settings.show_it:
            fig.show()
        if settings.save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)
        if settings.just_one:
            break


    else:
        for k,x in series_dict.items():
            print('not good')
    # print(result['fy'].max())
    


Index(['state_abrv', 'fy', 'year', 'total_enrollment',
       'american_indian_alaska_native', 'hispanic', 'black', 'white',
       'two_or_more', 'asian_pacific_islander'],
      dtype='object')
  state_abrv       year  fy  total_enrollment american_indian_alaska_native  asian_pacific_islander hispanic   black   white two_or_more
9         FL  2024-2025  25         2859655.0                          6698                   87753  1089035  597852  955470      122847
Index(['state_abrv', 'year', 'fy', 'total_enrollment',
       'american_indian_alaska_native', 'asian_pacific_islander', 'hispanic',
       'black', 'white', 'two_or_more'],
      dtype='object')
us
[np.int64(2940883), 14532747.0, 7352891.0, 21592113.0, 2522052.0]
state
[np.int64(87753), 1089035, 597852, 955470, 122847]
categories: ['Asian Pacific Islander', 'Hispanic', 'Black', 'White', 'Two Or More']
state vals: [87753.0, 1089035.0, 597852.0, 955470.0, 122847.0]
us vals: [2940883.0, 14532747.0, 7352891.0, 21592113.0, 25220

   state_abrv     year  fy  total_enrollment american_indian_alaska_native  asian_pacific_islander  hispanic   black     white two_or_more
45         UT  2023-24  24          689883.0                        6290.0                   22051  136346.0  9011.0  489791.0     26302.0
Index(['state_abrv', 'year', 'fy', 'total_enrollment',
       'american_indian_alaska_native', 'asian_pacific_islander', 'hispanic',
       'black', 'white', 'two_or_more'],
      dtype='object')
us
[np.int64(2940883), 14532747.0, 7352891.0, 21592113.0, 2522052.0]
state
[np.int64(22051), 136346.0, 9011.0, 489791.0, 26302.0]
categories: ['Asian Pacific Islander', 'Hispanic', 'Black', 'White', 'Two Or More']
state vals: [22051.0, 136346.0, 9011.0, 489791.0, 26302.0]
us vals: [2940883.0, 14532747.0, 7352891.0, 21592113.0, 2522052.0]
UT
STATE SUM
[22051.0, 136346.0, 9011.0, 489791.0, 26302.0]
683501.0
US SUM
[2940883.0, 14532747.0, 7352891.0, 21592113.0, 2522052.0]
48940686.0
STATE VALS
[3.2261840143613543, 19.948178

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv       year  fy  total_enrollment american_indian_alaska_native  asian_pacific_islander hispanic  black   white two_or_more
49         WI  2024-2025  25          805881.0                          7992                   35695   116817  70514  529890       44968
Index(['state_abrv', 'year', 'fy', 'total_enrollment',
       'american_indian_alaska_native', 'asian_pacific_islander', 'hispanic',
       'black', 'white', 'two_or_more'],
      dtype='object')
us
[np.int64(2940883), 14532747.0, 7352891.0, 21592113.0, 2522052.0]
state
[np.int64(35695), 116817, 70514, 529890, 44968]
categories: ['Asian Pacific Islander', 'Hispanic', 'Black', 'White', 'Two Or More']
state vals: [35695.0, 116817.0, 70514.0, 529890.0, 44968.0]
us vals: [2940883.0, 14532747.0, 7352891.0, 21592113.0, 2522052.0]
WI
STATE SUM
[35695.0, 116817.0, 70514.0, 529890.0, 44968.0]
797884.0
US SUM
[2940883.0, 14532747.0, 7352891.0, 21592113.0, 2522052.0]
48940686.0
STATE VALS
[4.473707957547714, 14.64085004837796, 

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 1.3 K-12 Enrollment, by Socioeconomic Status

In [16]:
#creates table 1.3 (example of 100% bar)
from graphs.viz_graphs_template import *
import plotly.graph_objects as go
import plotly.express as px

g_width,g_height = get_sizing('1.3')
axis_font_size = 45
# g_width = 2327
# g_height = 419
filename = f'1.3.png'
ses_data = data_pull.get_collected_data(metric = 'enr_ses')
ses_data.columns = [cleaning_col(x) for x in ses_data.columns]
# print(ses_data.to_string())
# data_pull.get_collected_data(get_sheets=True)

#graph settings
aspect_ratio = 7
# graph_h = 200
# graph_wd = graph_h*aspect_ratio

offset = 1500
multiplier = 1.5

#preset values before loop
categories = ['Economically Disadvantaged', 'Not Economically Disadvantaged']
us_values = ses_data[ses_data['state_abrv']=="US"]
# print('US only')
# print(us_values.to_string())

us_values = us_values.sort_values('year', ascending=False).reset_index(drop=True)
us_dict = us_values.to_dict(orient='records')[0]

# us_total
for jur in state_abbreviations_priority:
    if settings.subset_states != 'None' and jur not in settings.subset_states:
        continue
    if jur == 'US':
        continue
    # print(jur)
    state_name = state_ref_r.get(jur)
    #fetch for state
    result = ses_data[ses_data['state_abrv']==jur]
    result = result.loc[:,['state_abrv','year','percent_economically_disadvantaged']]
    
    # result = result[result['year']==result['year'].max()]
    res_dict = result.to_dict(orient='records')[0]
    # print(res_dict)
    # print(type(us_values))
    # print(us_values)
    dfs = {f'{jur}':result, "US":us_values}
    # concat_dfs = []

        
    fig = graph_1_3(res_dict, us_dict, jur, g_width, g_height)
    fig.update_yaxes(tickfont=get_base_text(axis_font_size))
    fig.update_layout(margin=dict(l=200, r=20, t=100, b=100))

    if settings.show_it:
        fig.show()
    if settings.save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)
    if settings.just_one:
            break


C:\Users\clutz\AppData\Local\Temp\ipykernel_10960\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_10960\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_10960\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 1.4: K-12 Enrollment, by Locale


In [16]:
#creates table 1.4 (example of 100% bar)



g_width,g_height = get_sizing('1.4')
axis_font_size = 45
# g_width = 2327
# g_height = 419
filename = f'1.4.png'
locale_data = data_pull.clean_enroll_by_locale()
locale_columns = list(locale_data.columns)
# print(locale_columns)
locale_data.columns = [cleaning_col(x) for x in locale_data.columns]
filter_state = None

# print(locale_data.to_string())
# for row in locale_data.itertuples(index = True):
#     print(row)
# data_pull.get_collected_data(get_sheets=True)


#graph settings
aspect_ratio = 7

offset = 1500
multiplier = 1.5

#preset values before loop

us_values = locale_data[locale_data['state_abrv']=="US"]
# print('US only')
# print(us_values.to_string())

us_dict = us_values.to_dict(orient='records')[0]

# us_total
for jur in state_abbreviations_priority:
    if settings.subset_states != 'None' and jur not in settings.subset_states:
        continue
    if jur == 'US':
        continue
    if settings.just_one == True and filter_state != None:
          if jur != filter_state:
                continue
    # print(jur)
    state_name = state_ref_r.get(jur)
    #fetch for state
    result = locale_data[locale_data['state_abrv']==jur]
    
    
    # result = result[result['year']==result['year'].max()]
    res_dict = result.to_dict(orient='records')[0]
    # print(res_dict)
    # print(us_dict)
    # print(type(us_values))
    # print(us_values)

    dfs = {f'{jur}':result, "US":us_values}
    # concat_dfs = []

        
    fig = graph_1_4(res_dict, us_dict, jur, g_width, g_height)
    fig.update_yaxes(tickfont=get_base_text(axis_font_size))
    fig.update_layout(margin=dict(l=200, r=20, t=100, b=100))

    if settings.show_it:
        fig.show()
    if settings.save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)
    if settings.just_one:
            break


united states
alabama
alaska
arizona
arkansas
california
colorado
connecticut
delaware
district of columbia
florida
georgia
hawaii
idaho
illinois
indiana
iowa
kansas
kentucky
louisiana
maine
maryland
massachusetts
michigan
minnesota
mississippi
missouri
montana
nebraska
nevada
new hampshire
new jersey
new mexico
new york
north carolina
north dakota
ohio
oklahoma
oregon
pennsylvania
rhode island
south carolina
south dakota
tennessee
texas
utah
vermont
virginia
washington
west virginia
wisconsin
wyoming
{'total': 2832516, 'city': 0.2727748051555578, 'suburban': 0.5433353244959604, 'town': 0.0443316118955727, 'rural': 0.13666754221335378}
{'total': 49089640, 'city': 0.29838493417348344, 'suburban': 0.3889082095529729, 'town': 0.10809665746173734, 'rural': 0.1996581152357198}


{'total': 687057, 'city': 0.1504984302612447, 'suburban': 0.5920236603367697, 'town': 0.11051630359635373, 'rural': 0.14578703077037278}
{'total': 49089640, 'city': 0.29838493417348344, 'suburban': 0.3889082095529729, 'town': 0.10809665746173734, 'rural': 0.1996581152357198}


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




{'total': 827381, 'city': 0.2785234372072842, 'suburban': 0.28818404096782496, 'town': 0.19141362927115801, 'rural': 0.24143048970184233}
{'total': 49089640, 'city': 0.29838493417348344, 'suburban': 0.3889082095529729, 'town': 0.10809665746173734, 'rural': 0.1996581152357198}


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 2

### 2.1: STATE ASSEMEMNT RESULTS

In [20]:
def reshape_education_data(df):
    """
    Reshapes education data from wide format to long format.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe with education data in wide format
        
    Returns:
    --------
    pandas.DataFrame
        Reshaped dataframe with columns: state_abrv, 2025_available, year, grade, subject, value
    """
    # Create a list to store all rows
    rows = []

    col_mapping = [
        (5, '2023', '4th grade', 'Reading'),
        (6, '2024', '4th grade', 'Reading'),
        (7, '2025', '4th grade', 'Reading'),
        (8, '2023', '4th grade', 'Math'),
        (9, '2024', '4th grade', 'Math'),
        (10, '2025', '4th grade', 'Math'),
        (11, '2023', '8th grade', 'Reading'),
        (12, '2024', '8th grade', 'Reading'),
        (13, '2025', '8th grade', 'Reading'),
        (14, '2023', '8th grade', 'Math'),
        (15, '2024', '8th grade', 'Math'),
        (16, '2025', '8th grade', 'Math'),
    ]
    
    # Iterate through each state (starting from row 3)
    for i in range(3, len(df)):
        state = df.iloc[i, 1]
        unavailable_flag = df.iloc[i, 4]
        
        # Determine if 2025 data is available
        # If the flag is True, then 2025 is unavailable, otherwise it's available
        data_2025_available = 'No' if unavailable_flag == True or (pd.notna(unavailable_flag) and str(unavailable_flag).strip().lower() == 'true') else 'Yes'
        
        # Iterate through each column mapping
        for col_idx, year, grade, subject in col_mapping:
            value = df.iloc[i, col_idx]
            
            # Skip if value is NaN or 0 (which seems to indicate missing data in your dataset)
            if pd.notna(value) and value != 0:
                rows.append({
                    'state_abrv': state,
                    '2025_available': data_2025_available,
                    'year': year,
                    'grade': grade,
                    'subject': subject,
                    'value': value
                })
    
    # Create the final dataframe
    result_df = pd.DataFrame(rows)
    
    # Sort by state, grade, subject, and year for better organization
    result_df = result_df.sort_values(['state_abrv', 'grade', 'subject', 'year']).reset_index(drop=True)
    
    return result_df

In [21]:
# GRAPHS for STATE ASSESSMENT RESULTS
g_width,g_height = get_sizing('2')

axis_font_size = 50

# g_width = 1352
# g_height = 768

state_assess_res = data_pull.get_collected_data(metric='state_assess_prof', no_header=True)
state_assess_res = state_assess_res.iloc[:,:17]
# cols = list(state_assess_res.columns)
# state_assess_res.columns = ['priority']+ cols[1:]
state_assess_res = state_assess_res[state_assess_res[0]!='x']


state_assess_res = reshape_education_data(state_assess_res)
state_assess_res = state_assess_res.sort_values(by=['state_abrv'])
state_assess_res = state_assess_res.sort_values(by=['grade','year','subject'], ascending=False).reset_index(drop=True)
grades = sorted(get_col_uniq_vals(state_assess_res['grade']))

# for state in state_abbreviations_priority:
#     if state == 'US':
#         continue
#     results = state_assess_res[state_assess_res['state_abrv']==state]

#     for grade in grades:
#         print(grade)
#         grad_res = results[results['grade']==grade]
#         print(grad_res.to_string())
#         fig = graph_2_state_assess_graph(grad_res, state)
#         fig.show()
break_loop = False
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
        continue
    if state == 'US':
        continue
    results = state_assess_res[state_assess_res['state_abrv']==state]

    for i,grade in enumerate(grades):
        print(grade)
        grad_res = results[results['grade']==grade]
        # Sort by year and subject to ensure consistent ordering
        grad_res = grad_res.sort_values(by=['year', 'subject']).reset_index(drop=True)
        print(grad_res.to_string())
        fig = graph_2_state_assess_graph(grad_res, state)
        fig.update_layout(
            width=g_width,
            height = g_height,
            xaxis = dict(tickfont=get_base_text(axis_font_size))
            )
        # fig.update_xaxes(
        #     tickvals=['2022-2023', '2023-2024', '2024-2025'],
        # )
                    

        fig.update_yaxes(
            visible=False)
        if settings.show_it:
            fig.show()
        if settings.save_it:
            save_to_folder(fig,f'2.{i+1}.png',g_width,g_height,state)
        if settings.just_one:
            break_loop=True
            break
    if break_loop == True:
        break
            

4th grade
  state_abrv 2025_available  year      grade  subject  value
0         UT            Yes  2023  4th grade     Math   49.6
1         UT            Yes  2023  4th grade  Reading   44.3
2         UT            Yes  2024  4th grade     Math   50.0
3         UT            Yes  2024  4th grade  Reading   44.0
4         UT            Yes  2025  4th grade     Math   49.1
5         UT            Yes  2025  4th grade  Reading   44.7


8th grade
  state_abrv 2025_available  year      grade  subject  value
0         UT            Yes  2023  8th grade     Math   39.1
1         UT            Yes  2023  8th grade  Reading   42.0
2         UT            Yes  2024  8th grade     Math   40.7
3         UT            Yes  2024  8th grade  Reading   42.5
4         UT            Yes  2025  8th grade     Math   41.1
5         UT            Yes  2025  8th grade  Reading   42.4


C:\Users\clutz\AppData\Local\Temp\ipykernel_20448\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_20448\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 3

### 3.1-4: NAEP Overall Proficiency Rates

In [19]:
#GRAPHS
g_width,g_height = get_sizing('3')
axis_font_size = 40
# g_width = 1340
# g_height = 768


naep_prof = data_pull.get_naep_data_long()
# print(naep_prof.sort_values(by='at_or_above_proficient', ascending=True).to_string())
us_only = naep_prof[naep_prof['state_abrv']=='US']

subsetting_states = []
offset = 7.7
break_loop = False
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
        continue
    if 'US' in state:
        continue
    print(bordered(state))
    result = naep_prof[naep_prof['state_abrv']==state]
    # print(result.to_string())
    graphs = result.loc[:,["grade", "subject"]]
    graphs = graphs.drop_duplicates(inplace=False).sort_values(by=['grade']).reset_index(drop=True)
    for row in graphs.itertuples():
        # retrieve data
        narrowed_results = result[(result['grade']==row.grade) & (result['subject']==row.subject)].sort_values(by='year').reset_index(drop = True)
        us_vals = us_only[(us_only['grade']==row.grade) & (us_only['subject']==row.subject)].sort_values(by='year').reset_index(drop = True)
        
        dfs_dict = {'state':narrowed_results,'us':us_vals}
        for k,v in dfs_dict.items():
            new = v.drop(labels = ['jurisdiction', 'grade','subject'], axis = 1)
            dfs_dict[k] = new
        
        # print(dfs_dict.get('us').head(2).to_string())
        # print(dfs_dict.get('state').head(2).to_string())
        merged = pd.concat(dfs_dict.values())
        # merged = pd.merge(dfs_dict.get('state'),dfs_dict.get('us'),on='year',how = 'outer', suffixes=('_state', '_us'))
        merged = merged.pivot(index = 'year', columns = 'state_abrv', values = 'at_or_above_proficient').reset_index()
        
        merged.columns = ['year','state', 'us']
        # df['col_3'] = df.apply(lambda row: f(row['col_1'], row['col_2']), axis=1)

        
        merged['state_position'] = merged.apply(lambda row: position_calc(row['state'],row['us'],multiplier=1.2), axis=1)
        merged['us_position'] = merged.apply(lambda row: position_calc(row['us'],row['state'],multiplier=1.2), axis=1)
        too_low = merged[(merged['state']<17)|(merged['us']<17)]
        
        if too_low.empty:
            y_min = 10
        else:
            y_min = 0
        # print('before')
        # print(merged.to_string())
        # for row_tl in too_low.itertuples(index=True):
        #     if row_tl.state <17:
        #         room = row_tl.us-row_tl.state
        #         max_room = room - 1
        #         if row_tl.state+1 > max_room:
        #             new_position = max_room+row_tl.state
        #         else:
        #             new_position = row_tl.state+1
        #         print(f'new_position: {new_position}')
        #         merged.loc[row_tl.Index,'state_position'] = new_position
                
        # print(too_low)
        print(f'{row.grade}th grade {row.subject.upper()}')
        print(merged.to_string())
        print(y_min)
        # merged['us_position'] = merged['us'].apply(lambda x: )

        fig = graph_naep_overall(merged,y_min, offset)
        
        # graph_naep_overall()
        if row.subject == 'reading':
            if row.grade == 4:
                graph_num = 1
            elif row.grade == 8:
                graph_num = 2

        if row.subject == 'math':
            if row.grade == 4:
                graph_num = 3
            elif row.grade == 8:
                graph_num = 4

        
        breathing = 60
        if graph_num == 1 or graph_num == 3:
            margins = dict(l=10, r=breathing, t=10, b=10)
        elif graph_num == 2 or graph_num == 4:
            margins = dict(l=breathing, r=10, t=10, b=10)
        fig.update_layout(
            width=g_width,
            height = g_height, 
            margin = margins,
            xaxis = dict(showline = True, linecolor = hunt_darkgray, tickfont=get_base_text(axis_font_size)),
            yaxis = dict(visible = False),
            )


            
        if settings.show_it:
            fig.show()
        if settings.save_it:
            save_to_folder(fig,f'3.{graph_num}.png',g_width,g_height,state)
        if settings.just_one:
            break_loop=True
            break
    if break_loop == True:
        break
            

┌──┐
│FL│
└──┘
4th grade MATH
   year      state         us  state_position  us_position
0  2011   37.33764  40.468978       31.337640    46.468978
1  2013  40.744145  41.816905       32.344145    50.216905
2  2015  42.025181  40.048913       50.425181    31.648913
3  2017  47.527704  40.194734       53.527704    34.194734
4  2019  47.530052  41.058182       53.530052    35.058182
5  2022  40.798326  36.267126       46.798326    30.267126
6  2024  44.777973  39.482018       50.777973    33.482018
10


4th grade READING
   year      state         us  state_position  us_position
0  2011  35.241768  33.709439       43.641768    25.309439
1  2013  38.955325  35.220014       44.955325    29.220014
2  2015  38.510167  36.107637       44.510167    30.107637
3  2017   40.81473  36.552006       46.814730    30.552006
4  2019  37.667877  35.341856       43.667877    29.341856
5  2022  38.990443  33.252708       44.990443    27.252708
6  2024  32.987005   31.14508       41.387005    22.745080
10


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




8th grade MATH
   year      state         us  state_position  us_position
0  2011  27.717959  34.735578       21.717959    40.735578
1  2013  30.754542  35.492217       24.754542    41.492217
2  2015  26.124443  33.142095       20.124443    39.142095
3  2017  29.150507  34.253203       23.150507    40.253203
4  2019  30.643313  33.849481       24.643313    39.849481
5  2022  22.948726  26.462034       16.948726    32.462034
6  2024  21.116375  28.029949       15.116375    34.029949
10


...

8th grade MATH
   year      state         us  state_position  us_position
0  2011  34.735578  40.970014       28.735578    46.970014
1  2013  35.492217  39.839877       29.492217    45.839877
2  2015  33.142095   40.78179       27.142095    46.781790
3  2017  34.253203  39.313459       28.253203    45.313459
4  2019  33.849481  41.305581       27.849481    47.305581
5  2022  26.462034  33.216641       20.462034    39.216641
6  2024  28.029949  36.527654       22.029949    42.527654
10


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




8th grade READING
   year      state         us  state_position  us_position
0  2011  33.504107  34.883244       25.104107    43.283244
1  2013  36.142239  36.458945       27.742239    44.858945
2  2015  34.313863  38.967398       28.313863    44.967398
3  2017  36.130515  39.388179       30.130515    45.388179
4  2019  33.578256    38.5285       27.578256    44.528500
5  2022  30.797971  32.394037       22.397971    40.794037
6  2024  29.841141  31.104695       21.441141    39.504695
10


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 4

### NAEP by region

In [1]:
#get the naep data
naep_df = data_pull.get_naep_data()
naep_df = naep_df[naep_df['state_abrv'].isnull()==False].sort_values(by='state_abrv').reset_index(drop=True)
print(naep_df.to_string())

NameError: name 'data_pull' is not defined

In [11]:
print(state_abbreviations)
print(state_abbreviations_priority)

['US', 'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'DC']
['AZ', 'CA', 'CT', 'DC', 'DE', 'FL', 'GA', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'MA', 'MD', 'MI', 'MN', 'MO', 'NC', 'ND', 'NE', 'NJ', 'NM', 'NY', 'OH', 'OK', 'OR', 'RI', 'SC', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']


In [ ]:
#region graphs dict setup
naep_data_dict = {}
for juri in state_abbreviations_priority+['US']:
    
    if juri == 'US':
        us_data = naep_df[naep_df['state_abrv']=="US"]
        naep_us_only = dict(tuple(us_data.groupby(['grade','subject'])))
        naep_data_dict[juri] = naep_us_only 
        # continue
    # print(juri)

    else:
        region_df = get_census_regions(juri)

        region_list = region_df['state_code'].to_list()
        region_data = naep_df[naep_df['state_abrv'].isin(region_list)]
    # print(region_data.to_string())

        # Split by state
        naep_by_grade_subject_df = dict(tuple(region_data.groupby(['grade','subject'])))
        naep_data_dict[juri] = naep_by_grade_subject_df
    # Access individual DataFrames
    
    # print(gr_4_math.to_string())
    # ak_df = dfs_by_state['AK']
    # for k,v in split_dfs.items():
    #     print(k,v)


In [ ]:
#graphs set up
g_width,g_height = get_sizing('4')

# g_width = 668
# g_height = 621

##### 4.1 NAEP Region_Grade 4 Reading


In [14]:
# GRAPHS: 4th grade reading

num = 1
offset_m = 1.25
axis_font_size = 37

filename = f'4.1.png'
us_naep_proficiency = naep_data_dict.get('US')[4,'reading']
# us_naep_proficiency = us_naep_proficiency[us_naep_proficiency['state_abrv']=='US'].reset_index(drop=True)
print(us_naep_proficiency.to_string())
for juri,df in naep_data_dict.items():
    if settings.subset_states != 'None' and juri not in settings.subset_states:
        continue
    if juri == 'US':
        continue

#
    print(juri)

    data_df = df[4,'reading'].sort_values(by='at_or_above_proficient', ascending=False)

    all_data = pd.concat([data_df,us_naep_proficiency]).reset_index(drop = True)
    all_data['color'] = all_data['state_abrv'].apply(lambda x: get_highlight_color(x, juri))
    
    # print(all_data.to_string())

    fig = graph_4_regions(all_data, offset_m, g_width, g_height,debug = False)
    fig.update_yaxes(
        tickfont=get_base_text(axis_font_size),
        )

    # fig.update_layout(margin = dict(l=20,t=10, r =0, b = 0))
    # fig.update_layout(margin = dict(l=200,t=0, r =10, b = 10))
    if settings.show_it:
        fig.show()
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)
    if settings.just_one:
        break


    state_abrv jurisdiction  year  grade  subject at_or_above_proficient
177         US     National  2024      4  reading               31.14508
UT
[30.0267476587493, 31.8712506480323, 31.8760671779882, 35.6550279312484, 35.8470806926919, 36.3221186565382, 31.145079696188]
45.40264832067275
['NV', 'ID', 'MT', 'CO', 'WY', 'UT', 'US']


C:\Users\clutz\AppData\Local\Temp\ipykernel_20448\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




##### 4.2 NAEP Region_Grade 4 Math 


In [15]:
# GRAPHS:  4th grade math
filename = f'4.2.png'
num = 1
offset_m = 1.25


us_naep_proficiency = naep_data_dict.get('US')[4,'math']
# us_naep_proficiency = us_naep_proficiency[us_naep_proficiency['state_abrv']=='US'].reset_index(drop=True)
print(us_naep_proficiency.to_string())
for juri,df in naep_data_dict.items():
    if settings.subset_states != 'None' and juri not in settings.subset_states:
        continue
    if juri == 'US':
        continue

    # print(juri)

    data_df = df[4,'math'].sort_values(by='at_or_above_proficient', ascending=False)

    all_data = pd.concat([data_df,us_naep_proficiency]).reset_index(drop = True)
    all_data['color'] = all_data['state_abrv'].apply(lambda x: get_highlight_color(x, juri))
    
    print(all_data.to_string())

    fig = graph_4_regions(all_data,offset_m, g_width, g_height)
    fig.update_yaxes(
        tickfont=get_base_text(axis_font_size))
    if settings.show_it:
        fig.show()
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)
    if settings.just_one:
        break
        

    state_abrv jurisdiction  year  grade subject at_or_above_proficient
179         US     National  2024      4    math              39.482018
  state_abrv jurisdiction  year  grade subject at_or_above_proficient    color
0         WY      Wyoming  2024      4    math              45.507528  #DCA3EB
1         UT         Utah  2024      4    math              45.190563  #500066
2         CO     Colorado  2024      4    math              41.637412  #DCA3EB
3         ID        Idaho  2024      4    math              40.749345  #DCA3EB
4         MT      Montana  2024      4    math              40.408382  #DCA3EB
5         NV       Nevada  2024      4    math              35.886094  #DCA3EB
6         AZ      Arizona  2024      4    math              33.806765  #DCA3EB
7         NM   New Mexico  2024      4    math              23.295787  #DCA3EB
8         US     National  2024      4    math              39.482018  #DCA3EB
[35.8860941390411, 40.4083818619266, 40.7493452796472, 41.63741221

C:\Users\clutz\AppData\Local\Temp\ipykernel_20448\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




##### 4.3 NAEP Region_Grade 8 Reading


In [16]:
#  GRAPHS: 8th grade reading
filename = f'4.3.png'
num = 1
offset_m = 1.25

us_naep_proficiency = naep_data_dict.get('US')[8,'reading']
# us_naep_proficiency = us_naep_proficiency[us_naep_proficiency['state_abrv']=='US'].reset_index(drop=True)
print(us_naep_proficiency.to_string())
for juri,df in naep_data_dict.items():
    if settings.subset_states != 'None' and juri not in settings.subset_states:
        continue
    if juri == 'US':
        continue

    # print(juri)

    data_df = df[8,'reading'].sort_values(by='at_or_above_proficient', ascending=False)

    all_data = pd.concat([data_df,us_naep_proficiency]).reset_index(drop = True)
    all_data['color'] = all_data['state_abrv'].apply(lambda x: get_highlight_color(x, juri))
    
    print(all_data.to_string())

    fig = graph_4_regions(all_data, offset_m,g_width, g_height)
    fig.update_yaxes(
        tickfont=get_base_text(axis_font_size))

    if settings.show_it:
        fig.show()
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)
    if settings.just_one:
        break


    state_abrv jurisdiction  year  grade  subject at_or_above_proficient
178         US     National  2024      8  reading              29.841141
  state_abrv jurisdiction  year  grade  subject at_or_above_proficient    color
0         CO     Colorado  2024      8  reading              34.792676  #DCA3EB
1         ID        Idaho  2024      8  reading               31.98298  #DCA3EB
2         UT         Utah  2024      8  reading              31.493964  #500066
3         MT      Montana  2024      8  reading              31.125059  #DCA3EB
4         WY      Wyoming  2024      8  reading               29.36048  #DCA3EB
5         NV       Nevada  2024      8  reading              25.978192  #DCA3EB
6         AZ      Arizona  2024      8  reading              25.399336  #DCA3EB
7         NM   New Mexico  2024      8  reading              18.537874  #DCA3EB
8         US     National  2024      8  reading              29.841141  #DCA3EB
[25.9781919912642, 29.3604798757942, 31.1250592537171,

C:\Users\clutz\AppData\Local\Temp\ipykernel_20448\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




##### 4.4 NAEP Region_Grade 8 Math


In [17]:
#  GRAPHS: 8th grade math
filename = f'4.4.png'
num = 1
offset_m = 1.25

us_naep_proficiency = naep_data_dict.get('US')[8,'math']
# us_naep_proficiency = us_naep_proficiency[us_naep_proficiency['state_abrv']=='US'].reset_index(drop=True)
print(us_naep_proficiency.to_string())
for juri,df in naep_data_dict.items():
    if settings.subset_states != 'None' and juri not in settings.subset_states:
        continue
    if juri == 'US':
        continue

    # print(juri)

    data_df = df[8,'math'].sort_values(by='at_or_above_proficient', ascending=False)

    all_data = pd.concat([data_df,us_naep_proficiency]).reset_index(drop = True)
    all_data['color'] = all_data['state_abrv'].apply(lambda x: get_highlight_color(x, juri))
    
    print(all_data.to_string())

    fig = graph_4_regions(all_data, offset_m,g_width, g_height)
    fig.update_yaxes(
        tickfont=get_base_text(axis_font_size))

    if settings.show_it:
        fig.show()
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)
    if settings.just_one:
        break



    state_abrv jurisdiction  year  grade subject at_or_above_proficient
176         US     National  2024      8    math              28.029949
  state_abrv jurisdiction  year  grade subject at_or_above_proficient    color
0         UT         Utah  2024      8    math              35.061626  #500066
1         CO     Colorado  2024      8    math               32.47483  #DCA3EB
2         MT      Montana  2024      8    math              32.097171  #DCA3EB
3         ID        Idaho  2024      8    math              30.915762  #DCA3EB
4         WY      Wyoming  2024      8    math              30.044219  #DCA3EB
5         AZ      Arizona  2024      8    math              25.794746  #DCA3EB
6         NV       Nevada  2024      8    math              20.199962  #DCA3EB
7         NM   New Mexico  2024      8    math              13.942698  #DCA3EB
8         US     National  2024      8    math              28.029949  #DCA3EB
[25.7947457905425, 30.0442193699283, 30.9157616994927, 32.09717057

C:\Users\clutz\AppData\Local\Temp\ipykernel_20448\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 4.5: State Assessment rigor


In [11]:
# data set up
state_cut_df = data_pull.get_naep_workbook_data(option='state cut')
state_cut_df = state_cut_df.iloc[4:,:].reset_index(drop=True)
state_cut_df.columns = state_cut_df.iloc[0,:]
state_cut_df = state_cut_df.iloc[1:,:].reset_index(drop=True)


# Rename first column to 'state'
state_cut_df.columns = ['state'] + list(state_cut_df.columns[1:])

# Melt the dataframe
result = []

for col in state_cut_df.columns[1:]:
    
    # Extract subject and year from column name
    parts = col.split()
    subject = parts[0]
    year = parts[1]
    
    # Create a temporary dataframe
    temp = state_cut_df[['state', col]].copy()
    temp.columns = ['state', 'value']
    temp['year'] = year
    temp['subject'] = subject
    
    result.append(temp)

# Concatenate all dataframes
final_df = pd.concat(result, ignore_index=True)

# Reorder columns
assess_rigor = final_df[['state', 'year', 'subject', 'value']]

# Replace em dashes and en dashes with NaN
assess_rigor['value'] = assess_rigor['value'].replace(['—', '–'], pd.NA)

# Convert value to numeric
assess_rigor['value'] = pd.to_numeric(assess_rigor['value'], errors='coerce')
assess_rigor['state_abrv'] = assess_rigor['state'].apply(get_state_abrv_from_lower)
popped = assess_rigor.pop('state_abrv')
assess_rigor.insert(0,'state_abrv', popped)

print(assess_rigor[assess_rigor['state_abrv']=='NC'].to_string())


    state_abrv           state  year  subject       value
33          NC  North Carolina  2022  reading  233.000000
84          NC  North Carolina  2022     math  249.000000
135         NC  North Carolina  2019  reading  231.894044
186         NC  North Carolina  2019     math  251.263531
237         NC  North Carolina  2017  reading  233.134416
288         NC  North Carolina  2017     math  241.749530
339         NC  North Carolina  2015  reading  231.876550
390         NC  North Carolina  2015     math  246.198062
441         NC  North Carolina  2013  reading  230.842211
492         NC  North Carolina  2013     math  247.722546
543         NC  North Carolina  2011  reading  203.078970
594         NC  North Carolina  2011     math  218.610334


In [12]:
def above_prof(x,prof):
    if x>= prof:
        return True
    else:
        return False
def apply_offset(x, offset, prof_bool):
    if prof_bool == True:
        return x+offset
    else:
        return x-offset

In [13]:
# GRAPHS 
g_width,g_height = get_sizing('4.5')

axis_font_size = 45
label_font_size = 45
# g_width = 1340
# g_height = 575
offset = 15
grade = 4
subjects = {'reading':'4.5.png','math':'4.6.png'}
break_loop = False
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
        continue
    if state == 'US':
        continue
    for sub,filename in subjects.items():
        # filename_plus = f'{sub}{filename}'
        result = assess_rigor[(assess_rigor['state_abrv']==state) & (assess_rigor['subject']==sub)].reset_index(drop=True).sort_values(by=['state_abrv','year'])
        ymin = min(result['value'])-50

        fig = graph_state_cut(result, 'value', state, result['year'],g_width, g_height)
        fig.update_yaxes(
            showticklabels=False,  # Hide tick labels
            showgrid=False,        # Hide gridlines
            zeroline=False         # Hide zero line
        )
        fig.update_layout(
            xaxis=dict(
                showgrid=False, 
                showline=True, 
                linecolor=hunt_darkgray, 
                tickmode='array', 
                tickvals=result['year'],
                tickfont=get_base_text(axis_font_size)

            ),
            yaxis=dict(
                range=[ymin, 275],
                showgrid=False, 
                showline=False, 
                linecolor=hunt_darkgray,
                visible = True),
            width = g_width,
            height= g_height)


        
        
            # Add text labels
        #get subj line for ref
        subj = get_col_uniq_vals(result['subject'])
        if len(subj)==1:
            if 'read' in str(subj[0]):
                subject_line = 238
            elif 'math' in str(subj[0]):
                subject_line = 249

        all_values = result['value'].to_list()
        val_2011 = all_values[0]
        y_max = max(all_values)
        y_min = min(all_values)
        yaxis_buffer = 1
        if not any(x > subject_line for x in all_values):
            height = subject_line-y_min
            rnge_y_max = subject_line+(height*yaxis_buffer)
            # continue
        else:
            height = y_max-y_min
            rnge_y_max = y_max+(height*yaxis_buffer)
        
        rnge_y_min = y_min-(height*yaxis_buffer)
        
        print(bordered(state))
        print(height)

        result['above_prof'] = result.apply(lambda row: above_prof(row['value'],subject_line), axis=1)
        text_labels = [subject_line]+ [''] * (len(result['year']) - 1)   # Show label only on last point
        # print(result.to_string())
        first_year = result[result['year'].astype(float)==2011].reset_index()
        # print(first_year)
        

        # if height<26:

        #     line_text_offset = 4
        # elif height >=26 and height <60:
        #     line_text_offset = 7
        # elif height >=60:
        line_text_offset = height*.2

        first_year_bool = first_year.at[0,'above_prof']
        # if first_year_bool != True:
        #     continue
        
        if first_year_bool == True:
            line_vals = [x-line_text_offset if x!=''else x for x in text_labels]
            text_labels = [f'{x}' if x!=''else x for x in text_labels]
        elif first_year_bool == False:
            line_vals = [x+line_text_offset if x!=''else x for x in text_labels]
            text_labels = [f'{x}' if x!=''else x for x in text_labels]
        
        offset = height*.32
        result['text_y_positions'] = result.apply(lambda row: apply_offset(row['value'],offset,row['above_prof']), axis=1)

        # print(result.to_string())
        # print(text_labels)
        text_y_positions = [float(x)-offset for x in result['value']]
        breathing = 70
        # Add text labels
        if '4.5' in filename:
            margins = dict(l=10, r=breathing, t=10, b=10)
        elif '4.6' in filename :
            margins = dict(l=breathing, r=10, t=10, b=10)

        # if abs(subject_line-val_2011)>10:
        #     continue


        #line label
        fig.add_trace(go.Scatter(
            x=result['year'].to_list(),
            y=line_vals,  # Use calculated positions
            mode='text',
            textposition='middle right',
            textfont=get_base_text(label_font_size, bold = True),
            text = text_labels,
            showlegend=False
        ))

        #data labels
        fig.add_trace(go.Scatter(
            x=result['year'].to_list(),
            y=result['text_y_positions'],  # Use calculated positions
            mode='text',
            showlegend=False,
            text=result['value'].apply(lambda x: str(int(round(x))) if pd.notna(x) else ''),
            textfont=get_base_text(45, bold = True)
        ))

        fig.update_layout(
            margin = margins,
            yaxis = dict(range = [rnge_y_min,rnge_y_max]))
        if settings.show_it:
            fig.show()
        if settings.save_it:
            # save_to_folder(fig,filename,g_width,g_height,juri)
            save_to_folder(fig,filename,g_width,g_height,state)
        # fig.show()
        # break
        if settings.just_one:
            break_loop = True
            break
    if break_loop == True:
        break


┌──┐
│UT│
└──┘
38.914406193793155


C:\Users\clutz\AppData\Local\Temp\ipykernel_7540\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│UT│
└──┘
29.714236272107


C:\Users\clutz\AppData\Local\Temp\ipykernel_7540\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 5

### 5.1-2 Proficiency %, by Race/Ethnicity


In [31]:
longitudal_data = data_pull.get_naep_workbook_data()
longitudal_data = longitudal_data.drop(2, axis=1)

longitudal_data.columns = [str(x).strip().replace('-','').replace(' ','_').lower() for x in longitudal_data.iloc[2,:]]
longitudal_data = longitudal_data.iloc[3:,:].reset_index(drop=True)


longitudal_data['state_abrv'] = longitudal_data['state'].apply(get_state_abrv_from_lower)
popped_col = longitudal_data.pop('state_abrv')
longitudal_data.insert(0,'state_abrv', popped_col)
columns = []
data_cols = []
for col in longitudal_data.columns:
    if 'proficiency' in str(col):
        grade_match = re.search(r'grade_(\d)', str(col))
        # print(grade_match)
        grade = grade_match.group(1)
        subj = col.split('_')[-1]
        col_name = f'{grade}th_{subj}_prof'
        columns.append(col_name)
        data_cols.append(col_name)
    else:
        columns.append(col)

longitudal_data.columns = columns
# print(longitudal_data.to_string())



In [32]:
#RE GRAPHS needs to be run for both subjects |scatter plot|
g_width,g_height = get_sizing('5')
g_width,g_height = get_sizing('5')
# g_width = 1142
# g_height = 541
#math or reading
subject = 'reading'
axis_font_size = 45



subjects = {'reading':'5.1.png','math':'5.2.png'}
break_loop = False
long_dat_dict = {}
for i,jur in enumerate(state_abbreviations_priority):
    if settings.subset_states != 'None' and jur not in settings.subset_states:
        continue
    if jur == "US":
        continue
    print(jur)
    # if i>10:
    #     break
    for sub,filename in subjects.items():
        result = longitudal_data[(longitudal_data['state_abrv']==jur)&(longitudal_data['year']>=2011)].reset_index(drop=True)
        print(result.to_string())
        # print(result[result['subgroup']=='Two or more races'].to_string())
        # for year in result['year']:
        #     print(type(year))
        years = sorted(get_col_uniq_vals(result['year']))
        values = sorted(result['year'])
        # x_range = [2011,2025]
        fig = graph_5_multi_series(result,f'4th_{sub}_prof','subgroup',years)
        
        fig.update_layout(
            width=g_width,
            height = g_height, 
            xaxis = dict(tickfont=get_base_text(axis_font_size)),
            margin = dict(l=10,t=10,r=10,b=10)
            # showlegend = True
        )
        fig.update_yaxes(visible = False)
        # full_filename = f'{sub}{filename}'
        # save_to_folder(fig,filename,g_width,g_height,jur)
        if settings.show_it:
            fig.show()
            
        if settings.save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)
        if settings.just_one:
                break_loop  = True
                break
        # fig.show()o
    if break_loop == True:
        break
    

FL
   state_abrv    state                       subgroup  year 4th_math_prof 4th_reading_prof 8th_math_prof 8th_reading_prof
0          FL  Florida                          White  2024            56               41            31               34
1          FL  Florida                          Black  2024            24               18             7               15
2          FL  Florida                       Hispanic  2024            43               31            15               21
3          FL  Florida         Asian/Pacific Islander  2024            78               51            56               55
4          FL  Florida  American Indian/Alaska Native  2024             ‡                ‡             ‡                ‡
5          FL  Florida              Two or more races  2024            41               44            32               26
6          FL  Florida                          White  2022            57               51            32               34
7          FL  Florid

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv    state                       subgroup  year 4th_math_prof 4th_reading_prof 8th_math_prof 8th_reading_prof
0          FL  Florida                          White  2024            56               41            31               34
1          FL  Florida                          Black  2024            24               18             7               15
2          FL  Florida                       Hispanic  2024            43               31            15               21
3          FL  Florida         Asian/Pacific Islander  2024            78               51            56               55
4          FL  Florida  American Indian/Alaska Native  2024             ‡                ‡             ‡                ‡
5          FL  Florida              Two or more races  2024            41               44            32               26
6          FL  Florida                          White  2022            57               51            32               34
7          FL  Florida  

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




UT
   state_abrv state                       subgroup  year 4th_math_prof 4th_reading_prof 8th_math_prof 8th_reading_prof
0          UT  Utah                          White  2024            53               42            42               35
1          UT  Utah                          Black  2024             ‡                ‡             ‡                ‡
2          UT  Utah                       Hispanic  2024            24               18            15               20
3          UT  Utah         Asian/Pacific Islander  2024            35                ‡             ‡                ‡
4          UT  Utah  American Indian/Alaska Native  2024             8                4            10               17
5          UT  Utah              Two or more races  2024            46               39             ‡               32
6          UT  Utah                          White  2022            50               43            41               41
7          UT  Utah                          

...

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




WI
   state_abrv      state                       subgroup  year 4th_math_prof 4th_reading_prof 8th_math_prof 8th_reading_prof
0          WI  Wisconsin                          White  2024            51               38            45               36
1          WI  Wisconsin                          Black  2024             5                8             7                9
2          WI  Wisconsin                       Hispanic  2024            24               20            17               20
3          WI  Wisconsin         Asian/Pacific Islander  2024            32               20            37               29
4          WI  Wisconsin  American Indian/Alaska Native  2024            18               18             8               15
5          WI  Wisconsin              Two or more races  2024            44               26            25               32
6          WI  Wisconsin                          White  2022            52               39            41               38
7    

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv      state                       subgroup  year 4th_math_prof 4th_reading_prof 8th_math_prof 8th_reading_prof
0          WI  Wisconsin                          White  2024            51               38            45               36
1          WI  Wisconsin                          Black  2024             5                8             7                9
2          WI  Wisconsin                       Hispanic  2024            24               20            17               20
3          WI  Wisconsin         Asian/Pacific Islander  2024            32               20            37               29
4          WI  Wisconsin  American Indian/Alaska Native  2024            18               18             8               15
5          WI  Wisconsin              Two or more races  2024            44               26            25               32
6          WI  Wisconsin                          White  2022            52               39            41               38
7       

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 5.3-4: Subgroup: Free/Reduced Lunch Elgibility (FRL)
table: 
NAEP Proficiency Rates by Free/Reduced Lunch | Grades 4 | 2009 - 2022

description:
The following graphs outline a breakdown of the percentage of students considered Proficient or better by Free/Reduced Lunch eligibility



In [33]:
frl_long_data = data_pull.get_naep_workbook_data(option='frl')
frl_long_data = frl_long_data.drop(3, axis=1)

frl_long_data.columns = [str(x).strip().replace('-','').replace(' ','_').lower() for x in frl_long_data.iloc[2,:]]
frl_long_data = frl_long_data.iloc[3:,:].reset_index(drop=True)


frl_long_data['state_abrv'] = frl_long_data['state'].apply(get_state_abrv_from_lower)
popped_col = frl_long_data.pop('state_abrv')
frl_long_data.insert(0,'state_abrv', popped_col)

frl_long_data.columns = ['state_abrv','year','state','eligibility','math_prof','reading_prof']
frl_long_data = frl_long_data[~frl_long_data['eligibility'].str.contains('information', case=False)]

# print(frl_long_data.to_string())


In [34]:
#FRL GRAPHS |scatter plot|

import math
g_width,g_height = get_sizing('5.3')
# g_width = 1142
# g_height = 541
#math or reading

axis_font_size = 45
subjects = {'reading':'5.3.png','math':'5.4.png'}
break_loop = False
frl_long_dat_dict = {}
for i,jur in enumerate(state_abbreviations_priority):
    if settings.subset_states != 'None' and jur not in settings.subset_states:
        continue
    if jur == "US":
        continue

    for sub, filename in subjects.items():
        result = frl_long_data[(frl_long_data['state_abrv']==jur)&(frl_long_data['year']>=2011)].reset_index(drop=True)
        # print(result.to_string())
        years = sorted(get_col_uniq_vals(result['year']))
        values = sorted(result['year'])
        
        
        print(bordered(jur))
        if sub =='reading':
            dropping = 'math_prof'
            keeping = 'reading_prof'
        elif sub == 'math':
            dropping = 'reading_prof'
            keeping = 'math_prof'

        result_s = result.drop(labels=dropping, axis = 1)
        # result_s[keeping] = result_s[keeping].where(result_s[keeping].str.contains('‡'))
        result_s.loc[result_s[keeping].astype(str).str.contains('‡'), keeping] = np.nan

        df_pivoted = result_s.pivot(
        index=['state_abrv', 'year', 'state'],
        columns='eligibility',
        values=keeping
        ).reset_index()
        df_pivoted.columns = [x.strip().replace(' ','_').lower() for x in df_pivoted.columns]

        
        all_values = df_pivoted['eligible'].to_list()+df_pivoted['not_eligible'].to_list()
        all_values = [x for x in all_values if '‡' not in str(x)]
        all_values = [x for x in all_values if math.isnan(float(x))==False]
        y_max = max(all_values)
        y_min = min(all_values)
        # print(y_max)
        rnge_y_max = y_max*1.3
        rnge_y_min = y_min*(-1.3)
        height = rnge_y_max-rnge_y_min
        
        df_pivoted = df_pivoted.replace('‡', )
        df_pivoted['el_position'] = df_pivoted.apply(lambda row: get_val_offset(row['eligible'], row['not_eligible'], height*.10), axis = 1)
        df_pivoted['not_position'] = df_pivoted.apply(lambda row: get_val_offset(row['not_eligible'],row['eligible'], height*.10), axis = 1)
        df_pivoted['el_position'] = df_pivoted['el_position'].ffill()
        df_pivoted['not_position'] = df_pivoted['not_position'].ffill()
        # print(df_pivoted.to_string(max_colwidth=30))
        # df_pivoted['eligible'] = [x if math.isnan(float(x))==False or '' else '' for x in df_pivoted['eligible']]
        # df_pivoted['not_eligible'] = [float(x) if math.isnan(float(x))==False else '' for x in df_pivoted['not_eligible']]
        df_pivoted['eligible'] = [f"{x:.0f}%" if isinstance(x, (int, float)) and 'nan' not in str(x).lower() else '' for x in df_pivoted['eligible']]
        df_pivoted['not_eligible'] = [f"{x:.0f}%" if isinstance(x, (int, float)) and 'nan' not in str(x).lower() else '' for x in df_pivoted['not_eligible']]

        # print(df_pivoted.to_string(max_colwidth=30))
        # print(result_math.to_string())
        # print(result_reading.to_string())
        # print(result.to_string())
        # for year in result['year']:
        #     print(type(year))
        # print(all_values)


        # x_range = [2011,2025]
        fig = graph_5_multi_series(result,f'{sub}_prof','eligibility',years, label_all=True)
         # Add text labels
        fig.add_trace(go.Scatter(
            x=df_pivoted['year'],
            y=df_pivoted['el_position'],
            mode='text',
            showlegend=False,
            text=df_pivoted['eligible'],
            textfont=get_base_text(font_size_graph, bold=True),
            textposition="middle center",  # Use the pos variable you calculated
            texttemplate=df_pivoted['eligible']
            ))
        fig.add_trace(go.Scatter(
            x=df_pivoted['year'].to_list(),
            y=df_pivoted['not_position'],
            mode='text',
            showlegend=False,
            text=df_pivoted['not_eligible'],
            textfont=get_base_text(font_size_graph, bold=True),
            textposition="middle center",  # Use the pos variable you calculated
            texttemplate=df_pivoted['not_eligible']
            ))
        breathing = 60
        if '5.3' in filename:
            margins = dict(l=10, r=breathing, t=10, b=10)
        elif '5.4' in filename :
            margins = dict(l=breathing, r=10, t=10, b=10)
        fig.update_layout(
            width=g_width,
            height = g_height, 
            margin = margins,
            # -showlegend = True,
            
            xaxis=dict(tickfont = get_base_text(axis_font_size)))
        fig.update_yaxes(
            range=[rnge_y_min, rnge_y_max],
            tickvals=list(range(0, 90, 10)),
            ticksuffix='%',
            showgrid=False,
            showline=True,
            linecolor='black',
            visible = False
        )
        
        if settings.show_it:
            fig.show()
        if settings.just_one:
            break_loop = True
            break
        if settings.save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)
    if break_loop == True:
        break


┌──┐
│FL│
└──┘


┌──┐
│FL│
└──┘


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│UT│
└──┘


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




...

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│WI│
└──┘


┌──┐
│WI│
└──┘


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 5.5-6: Subgroup: English Language Learners (ELL)

table: 
NAEP Proficiency Rates for English Language Learners | Grades 4 | 2010 - 2024

description:
The following graphs outline a breakdown of the percentage of students considered Proficient or better English Language Learners status



In [35]:
ell_long_data = data_pull.get_naep_workbook_data(option='ell')
# ell_long_data = ell_long_data.drop(3, axis=1)

ell_long_data.columns = [str(x).strip().replace('-','').replace(' ','_').replace('/jurisdiction','').lower() for x in ell_long_data.iloc[2,:]]
ell_long_data = ell_long_data.iloc[3:,:].reset_index(drop=True)


ell_long_data['state_abrv'] = ell_long_data['state/jurisdiction'].apply(get_state_abrv_from_lower)
popped_col = ell_long_data.pop('state_abrv')
ell_long_data.insert(0,'state_abrv', popped_col)

ell_long_data.columns = ['state_abrv','year','state','ell_status','math_prof','reading_prof']
ell_long_data = ell_long_data[~ell_long_data['ell_status'].str.contains('information', case=False)]
# print(ell_long_data.to_string())



In [36]:
# ELL GRAPHS |scatter plot|
#math or reading
# subject="math"
# filename = f'5.4.png'

g_width,g_height = get_sizing('5.3')
axis_font_size = 45

# g_width = 1142
# g_height = 541
subjects = {'reading':'5.5.png','math':'5.6.png'}

ell_long_dat_dict = {}
break_loop = False
for i, jur in enumerate(state_abbreviations_priority):
    if settings.subset_states != 'None' and jur not in settings.subset_states:
        continue
    if jur == "US":
        continue
    
    for sub, filename in subjects.items():
        result = ell_long_data[(ell_long_data['state_abrv']==jur) & (ell_long_data['year']>=2011)].reset_index(drop=True)
        # print(result.to_string())
        print(jur)
        if sub =='reading':
            dropping = 'math_prof'
            keeping = 'reading_prof'
        elif sub == 'math':
            dropping = 'reading_prof'
            keeping = 'math_prof'

        result_s = result.drop(labels=dropping, axis = 1)
        df_pivoted = result_s.pivot(
        index=['state_abrv', 'year', 'state'],
        columns='ell_status',
        values=keeping
        ).reset_index()
        df_pivoted.columns = [x.strip().replace(' ','_').lower() for x in df_pivoted.columns]

        
        all_values = df_pivoted['ell'].to_list()+df_pivoted['not_ell'].to_list()
        all_values = [x for x in all_values if '#' not in str(x)]
        all_values = [x for x in all_values if '‡' not in str(x)]
        all_values = [x for x in all_values if math.isnan(float(x))==False]
        y_max = max(all_values)
        y_min = min(all_values)
        
        # print('___________')
        # print('True Range')
        # print(y_min)
        # print(y_max)
        # print(y_max)
        rnge_y_max = y_max*1.5
        if y_min <=2:
            rnge_y_min = y_min-15
        elif y_min <=10:
            rnge_y_min = y_min-15
        else:
            rnge_y_min = y_min*(-2)
        height = rnge_y_max-rnge_y_min
        # print(rnge_y_min, rnge_y_max)

        df_pivoted['ell'] = pd.to_numeric(df_pivoted['ell'], errors='coerce')
        df_pivoted['not_ell'] = pd.to_numeric(df_pivoted['not_ell'], errors='coerce')
        
        df_pivoted['el_position'] = df_pivoted.apply(lambda row: get_val_offset(row['ell'], row['not_ell'], height*.13), axis = 1)
        df_pivoted['not_position'] = df_pivoted.apply(lambda row: get_val_offset(row['not_ell'],row['ell'], height*.13), axis = 1)
        
        
        df_pivoted['el_position'] = df_pivoted['el_position'].ffill()
        df_pivoted['not_position'] = df_pivoted['not_position'].ffill()
        # print(df_pivoted.to_string(max_colwidth=30))
        df_pivoted['ell'] = [f"{x:.0f}%" if isinstance(x, (int, float)) and 'nan' not in str(x).lower() else '' for x in df_pivoted['ell']]
        df_pivoted['not_ell'] = [f"{x:.0f}%" if isinstance(x, (int, float)) and 'nan' not in str(x).lower() else '' for x in df_pivoted['not_ell']]


        # print(df_pivoted.to_string())
        # print(result_math.to_string())
        # print(result_reading.to_string())
        # print(result.to_string())
        # for year in result['year']:
        #     print(type(year))
        # print(all_values)

        # x_range = [2011,2025]
        
        fig = graph_5_multi_series(result, f'{sub}_prof', 'ell_status', result['year'], label_all=True)
         # Add text labels
        fig.add_trace(go.Scatter(
            x=df_pivoted['year'],
            y=df_pivoted['el_position'],
            mode='text',
            showlegend=False,
            text=df_pivoted['ell'],
            textfont=get_base_text(font_size_graph, bold=True),
            textposition="middle center",  # Use the pos variable you calculated
            texttemplate=df_pivoted['ell']
            ))
        fig.add_trace(go.Scatter(
            x=df_pivoted['year'].to_list(),
            y=df_pivoted['not_position'],
            mode='text',
            showlegend=False,
            text=df_pivoted['not_ell'],
            textfont=get_base_text(font_size_graph, bold=True),
            textposition="middle center",  # Use the pos variable you calculated
            texttemplate=df_pivoted['not_ell']
            ))
            # print(df_pivoted.to_string())
        years = sorted(get_col_uniq_vals(result['year']))
        
        breathing = 60
        if '5.5' in filename:
            margins = dict(l=10, r=breathing, t=10, b=10)
        elif '5.6' in filename :
            margins = dict(l=breathing, r=10, t=10, b=10)
        fig.update_layout(
            width=g_width,
            margin = margins,
            height=g_height,
            xaxis=dict(tickfont = get_base_text(axis_font_size))
        )
        
        # Override yaxis separately
        fig.update_yaxes(
            range=[rnge_y_min, rnge_y_max],
            tickvals=list(range(0, 90, 10)),
            ticksuffix='%',
            showgrid=False,
            showline=True,
            linecolor='black',
            visible = False
        )
        
        if settings.show_it:
            fig.show()
            if settings.just_one:
                break_loop = True
                break
        if settings.save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)

    if break_loop == True:
        break

    

FL


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




FL


UT


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




...

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




WI


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




WI


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 5.7-8: Subgroup: Disability Status (SWD)

table: 
NAEP Proficiency Rates for Students with Disabilities | Grades 4 | 2010 - 2024

description:
The following graphs outline a breakdown of the percentage of students considered Proficient or better by disability status


In [37]:
swd_long_data = data_pull.get_naep_workbook_data(option='swd')
# swd_long_data = swd_long_data.drop(3, axis=1)

swd_long_data.columns = [str(x).strip().replace('-','').replace(' ','_').replace('/jurisdiction','').lower() for x in swd_long_data.iloc[2,:]]
swd_long_data = swd_long_data.iloc[3:,:].reset_index(drop=True)


swd_long_data['state_abrv'] = swd_long_data['state/jurisdiction'].apply(get_state_abrv_from_lower)
popped_col = swd_long_data.pop('state_abrv')
swd_long_data.insert(0,'state_abrv', popped_col)

swd_long_data.columns = ['state_abrv','year','state','dis_status','math_prof','reading_prof']
swd_long_data = swd_long_data[~swd_long_data['dis_status'].str.contains('information', case=False)]
# print(swd_long_data.to_string())



In [38]:
#SWD GRAPHS |scatter plot|

g_width,g_height = get_sizing('5.7')

# g_width = 1142
# g_height = 541
#math or reading
axis_font_size = 45

subjects = {'reading':'5.7.png','math':'5.8.png'}
break_loop = False
swd_long_dat_dict = {}
for i,jur in enumerate(state_abbreviations_priority):
    if settings.subset_states != 'None' and jur not in settings.subset_states:
        continue
    if jur == "US":
        continue

    for sub,filename in subjects.items():
        
        result = swd_long_data[(swd_long_data['state_abrv']==jur)&(swd_long_data['year']>=2011)].reset_index(drop=True)
        print(result.to_string())
        # for year in result['year']:
        #     print(type(year))
        years = sorted(get_col_uniq_vals(result['year']))
        values = sorted(result['year'])
        if sub =='reading':
            dropping = 'math_prof'
            keeping = 'reading_prof'
        elif sub == 'math':
            dropping = 'reading_prof'
            keeping = 'math_prof'

        result_s = result.drop(labels=dropping, axis = 1)
        df_pivoted = result_s.pivot(
        index=['state_abrv', 'year', 'state'],
        columns='dis_status',
        values=keeping
        ).reset_index()
        df_pivoted.columns = [x.strip().replace(' ','_').lower() for x in df_pivoted.columns]

        
        all_values = df_pivoted['identified_as_students_with_disabilities'].to_list()+df_pivoted['not_identified_as_students_with_disabilities'].to_list()
        all_values = [x for x in all_values if '‡' not in str(x)]
        all_values = [x for x in all_values if math.isnan(float(x))==False]
        y_max = max(all_values)
        y_min = min(all_values)
        # print(y_max)
        rnge_y_max = y_max*1.4
        rnge_y_min = y_min*(-1.4)
        height = rnge_y_max-rnge_y_min

        df_pivoted['el_position'] = df_pivoted.apply(lambda row: get_val_offset(row['identified_as_students_with_disabilities'], row['not_identified_as_students_with_disabilities'], height*.12), axis = 1)
        df_pivoted['not_position'] = df_pivoted.apply(lambda row: get_val_offset(row['not_identified_as_students_with_disabilities'],row['identified_as_students_with_disabilities'], height*.12), axis = 1)
        
        
        df_pivoted['el_position'] = df_pivoted['el_position'].ffill()
        df_pivoted['not_position'] = df_pivoted['not_position'].ffill()
        # print(df_pivoted.to_string(max_colwidth=30))
        # df_pivoted['eligible'] = [x if math.isnan(float(x))==False or '' else '' for x in df_pivoted['eligible']]
        # df_pivoted['not_eligible'] = [float(x) if math.isnan(float(x))==False else '' for x in df_pivoted['not_eligible']]
        df_pivoted['identified_as_students_with_disabilities'] = [f"{x:.0f}%" if isinstance(x, (int, float)) and 'nan' not in str(x).lower() else '' for x in df_pivoted['identified_as_students_with_disabilities']]
        df_pivoted['not_identified_as_students_with_disabilities'] = [f"{x:.0f}%" if isinstance(x, (int, float)) and 'nan' not in str(x).lower() else '' for x in df_pivoted['not_identified_as_students_with_disabilities']]

        
        print(df_pivoted.to_string())
        # print(result_math.to_string())
        # print(result_reading.to_string())
        # print(result.to_string())
        # for year in result['year']:
        #     print(type(year))
        # print(all_values)

        # x_range = [2011,2025]
        
        fig = graph_5_multi_series(result,f'{sub}_prof','dis_status',years, label_all=True)
         # Add text labels
        fig.add_trace(go.Scatter(
            x=df_pivoted['year'],
            y=df_pivoted['el_position'],
            mode='text',
            showlegend=False,
            text=df_pivoted['identified_as_students_with_disabilities'],
            textfont=get_base_text(font_size_graph, bold=True),
            textposition="middle center",  # Use the pos variable you calculated
            texttemplate=df_pivoted['identified_as_students_with_disabilities']
            ))
        fig.add_trace(go.Scatter(
            x=df_pivoted['year'].to_list(),
            y=df_pivoted['not_position'],
            mode='text',
            showlegend=False,
            text=df_pivoted['not_identified_as_students_with_disabilities'],
            textfont=get_base_text(font_size_graph, bold=True),
            textposition="middle center",  # Use the pos variable you calculated
            texttemplate=df_pivoted['not_identified_as_students_with_disabilities']
            ))
        
        
        # x_range = [2011,2025]
        
        breathing = 60
        if '5.7' in filename:
            margins = dict(l=10, r=breathing, t=10, b=10)
        elif '5.8' in filename :
            margins = dict(l=breathing, r=10, t=10, b=10)


        fig.update_layout(
                width=g_width,
                height=g_height,
                margin = margins,
                xaxis=dict(tickfont = get_base_text(axis_font_size))# font=get_base_text(axis_font_size)
            )
            
        # Override yaxis separately to ensure it takes effect
        fig.update_yaxes(
            range=[rnge_y_min, rnge_y_max],
            tickvals=list(range(0, 90, 10)),
            ticksuffix='%',
            showgrid=False,
            showline=True,
            linecolor='black',
            visible = False
    
        )
        if settings.show_it:
            fig.show()
        if settings.just_one:
            break_loop = True
            break
        if settings.save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)

    if break_loop == True:
        break

   state_abrv  year    state                                    dis_status math_prof reading_prof
0          FL  2024  Florida      Identified as students with disabilities        18           15
1          FL  2024  Florida  Not identified as students with disabilities        51           37
2          FL  2022  Florida      Identified as students with disabilities        22           19
3          FL  2022  Florida  Not identified as students with disabilities        45           44
4          FL  2019  Florida      Identified as students with disabilities        26           16
5          FL  2019  Florida  Not identified as students with disabilities        53           43
6          FL  2017  Florida      Identified as students with disabilities        26           19
7          FL  2017  Florida  Not identified as students with disabilities        52           45
8          FL  2015  Florida      Identified as students with disabilities        23           16
9          FL  2015 

   state_abrv  year    state                                    dis_status math_prof reading_prof
0          FL  2024  Florida      Identified as students with disabilities        18           15
1          FL  2024  Florida  Not identified as students with disabilities        51           37
2          FL  2022  Florida      Identified as students with disabilities        22           19
3          FL  2022  Florida  Not identified as students with disabilities        45           44
4          FL  2019  Florida      Identified as students with disabilities        26           16
5          FL  2019  Florida  Not identified as students with disabilities        53           43
6          FL  2017  Florida      Identified as students with disabilities        26           19
7          FL  2017  Florida  Not identified as students with disabilities        52           45
8          FL  2015  Florida      Identified as students with disabilities        23           16
9          FL  2015 

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv  year state                                    dis_status math_prof reading_prof
0          UT  2024  Utah      Identified as students with disabilities        23           12
1          UT  2024  Utah  Not identified as students with disabilities        49           41
2          UT  2022  Utah      Identified as students with disabilities        16           12
3          UT  2022  Utah  Not identified as students with disabilities        47           42
4          UT  2019  Utah      Identified as students with disabilities        19           16
5          UT  2019  Utah  Not identified as students with disabilities        50           44
6          UT  2017  Utah      Identified as students with disabilities        19           16
7          UT  2017  Utah  Not identified as students with disabilities        49           45
8          UT  2015  Utah      Identified as students with disabilities        20           12
9          UT  2015  Utah  Not identified as stude

...

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv  year      state                                    dis_status math_prof reading_prof
0          WI  2024  Wisconsin      Identified as students with disabilities        18           11
1          WI  2024  Wisconsin  Not identified as students with disabilities        47           35
2          WI  2022  Wisconsin      Identified as students with disabilities        21           10
3          WI  2022  Wisconsin  Not identified as students with disabilities        47           36
4          WI  2019  Wisconsin      Identified as students with disabilities        14           12
5          WI  2019  Wisconsin  Not identified as students with disabilities        49           39
6          WI  2017  Wisconsin      Identified as students with disabilities        16           11
7          WI  2017  Wisconsin  Not identified as students with disabilities        46           39
8          WI  2015  Wisconsin      Identified as students with disabilities        17           13


   state_abrv  year      state                                    dis_status math_prof reading_prof
0          WI  2024  Wisconsin      Identified as students with disabilities        18           11
1          WI  2024  Wisconsin  Not identified as students with disabilities        47           35
2          WI  2022  Wisconsin      Identified as students with disabilities        21           10
3          WI  2022  Wisconsin  Not identified as students with disabilities        47           36
4          WI  2019  Wisconsin      Identified as students with disabilities        14           12
5          WI  2019  Wisconsin  Not identified as students with disabilities        49           39
6          WI  2017  Wisconsin      Identified as students with disabilities        16           11
7          WI  2017  Wisconsin  Not identified as students with disabilities        46           39
8          WI  2015  Wisconsin      Identified as students with disabilities        17           13


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 6

### 6.1 chronic absenteesism (Race)

In [39]:
# set up
chron_abs_df = data_pull.get_collected_data(metric='chron_abs_race', no_header = True)
chron_abs_df = chron_abs_df.iloc[1:, 1:13].reset_index(drop = True)
chron_abs_df = chron_abs_df.drop(columns=chron_abs_df.columns[[1,2,4]])
# print(chron_abs_df.head().to_string(max_colwidth=30))
# Set proper column names
chron_abs_df.columns = [
    'state_abrv', 'year', 'White', 'Black', 'Hispanic', 'Asian',
    'American Indian/Alaska Native', 'Native Hawaiian/Other Pacific Islander', 'Two or More'
]
chron_abs_df = chron_abs_df[chron_abs_df['state_abrv']!='State']





# Clean year column to pick later year if range
def extract_later_year(val):
    if pd.isna(val):
        return np.nan
    years = re.findall(r'\d{4}', str(val))
    return int(years[-1]) if years else np.nan

chron_abs_df['year'] = chron_abs_df['year'].apply(extract_later_year)

# Melt into long format
df_long = chron_abs_df.melt(
    id_vars=['state_abrv', 'year'],
    var_name='group',
    value_name='value'
)

# Normalize values to percentages
def normalize_percentage(x):
    try:
        x = float(str(x).replace('%','').replace('%%',''))  # remove stray % symbols
        if x > 1.5:  # likely already in percent form
            return x
        else:        # decimal → percent
            return x * 100
    except:
        return np.nan

df_long['value'] = df_long['value'].apply(normalize_percentage)

# Optional: sort for readability
chron_abs_df = df_long.sort_values(['state_abrv', 'group']).reset_index(drop=True)

# Preview
# print(chron_abs_df.to_string(max_colwidth=30))


In [40]:
# GRAPHS chronic absenteeism |bar vertical|
# g_width = 2703
# g_height = 628
g_width,g_height = get_sizing('6.1')
axis_font_size = 38
offset = 1
filename = f'6.1.png'
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
        continue
    if state == 'US':
        continue
    result = chron_abs_df[chron_abs_df['state_abrv']==state].reset_index(drop=True)
    result.loc[result['group']=='Two or More','group'] = 'Two or More Races'
    print('result')
    print(result.to_string(max_colwidth=30))

    
    fig = graph_chron_abs(result, 'value', g_width, g_height)
    fig.update_layout(
        xaxis=dict(tickfont=get_base_text(axis_font_size), labelalias = {'Two or More':'Two or More Races'})
    )
    fig.update_yaxes(
        visible=False
    )
    if settings.show_it:
        fig.show()
        if settings.just_one:
            break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)

    


result
  state_abrv    year                          group  value
0         FL  2024.0  American Indian/Alaska Native    0.3
1         FL  2024.0                          Asian    1.4
2         FL  2024.0                          Black   24.9
3         FL  2024.0                       Hispanic   39.7
4         FL  2024.0  Native Hawaiian/Other Paci...    0.2
5         FL  2024.0              Two or More Races    4.2
6         FL  2024.0                          White   29.5
                           group
0  American Indian/Alaska Native
1  Native Hawaiian/Other Paci...
2                          Asian
3                       Hispanic
4                          Black
5                          White
6              Two or More Races
looking at groups 
  state_abrv    year                          group  value
0         FL  2024.0  American Indian/Alaska Native    0.3
1         FL  2024.0                          Asian    1.4
2         FL  2024.0                          Black   24.9
3 

result
  state_abrv    year                          group  value
0         UT  2024.0  American Indian/Alaska Native   39.0
1         UT  2024.0                          Asian   17.0
2         UT  2024.0                          Black   26.0
3         UT  2024.0                       Hispanic   32.0
4         UT  2024.0  Native Hawaiian/Other Paci...   42.0
5         UT  2024.0              Two or More Races   25.0
6         UT  2024.0                          White   19.0
                           group
0  American Indian/Alaska Native
1  Native Hawaiian/Other Paci...
2                          Asian
3                       Hispanic
4                          Black
5                          White
6              Two or More Races
looking at groups 
  state_abrv    year                          group  value
0         UT  2024.0  American Indian/Alaska Native   39.0
1         UT  2024.0                          Asian   17.0
2         UT  2024.0                          Black   26.0
3 

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




result
  state_abrv    year                          group  value
0         WI  2024.0  American Indian/Alaska Native   40.0
1         WI  2024.0                          Asian   13.1
2         WI  2024.0                          Black   47.9
3         WI  2024.0                       Hispanic   27.4
4         WI  2024.0  Native Hawaiian/Other Paci...   24.7
5         WI  2024.0              Two or More Races   23.7
6         WI  2024.0                          White   11.2
                           group
0  American Indian/Alaska Native
1  Native Hawaiian/Other Paci...
2                          Asian
3                       Hispanic
4                          Black
5                          White
6              Two or More Races
looking at groups 
  state_abrv    year                          group  value
0         WI  2024.0  American Indian/Alaska Native   40.0
1         WI  2024.0                          Asian   13.1
2         WI  2024.0                          Black   47.9
3 

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 6.2 chronic absenteesism (Other Subgroup)

In [41]:
# chronic absenteesism other
chron_abs_df_other = data_pull.get_collected_data(metric='chron_abs_other', no_header = True)
chron_abs_df_other = chron_abs_df_other.drop(columns=chron_abs_df_other.columns[1:6])

chron_abs_df_other.columns = ['state', 'year']+list(chron_abs_df_other.iloc[1,2:])
chron_abs_df_other = chron_abs_df_other.iloc[2:, :].reset_index(drop=True)
chron_abs_df_other = chron_abs_df_other.drop(columns=['notes', 'date pulled'])
print(chron_abs_df_other.columns)
# print(chron_abs_df_other.to_string())


Index(['state', 'year', 'All', 'Economically Disadvantaged',
       'English Language Learners', 'Students With Disabilities'],
      dtype='object')


In [42]:
# GRAPHS chronic absenteeism other |bar vertical|
g_width,g_height = get_sizing('6.2')
axis_font_size = 45
filename = '6.2.png'
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    result = chron_abs_df_other[chron_abs_df_other['state']==state].reset_index()
    print(result.to_string())
    fig = graph_other_chron_abs(result)
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=get_base_text(axis_font_size)
    )
    fig.update_yaxes(
        visible=False
    )
    if settings.show_it:
        fig.show()
        if settings.just_one:
            break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)

   index state                                                                               year    All Economically Disadvantaged English Language Learners Students With Disabilities
0      9    FL  https://eddataexpress.ed.gov/resources/reports-and-files/chronic-absenteeism-data  0.338                        NaN                     0.123                       0.19
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   index state                                                                                                  year    All Economically Disadvantaged English Language Learners Students With Disabilities
0     44    UT  https://reportcard.schools.utah.gov/State/Unscored/?StateID=99&SchoolLevel=HS&schoolyearendyear=2024  0.225                       0.34                      0.36                        0.3
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']


   index state                                                   year   All Economically Disadvantaged English Language Learners Students With Disabilities
0     49    WI  https://wisedash.dpi.wi.gov/Dashboard/dashboard/16640  17.7                      0.295                     0.252                      0.265
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 6.3 Out of school suspension

In [43]:
#out of school suspension
oos_df = data_pull.get_collected_data(metric='suspension_race', no_header = True)
oos_df.columns = list(oos_df.iloc[1,:])
oos_df = oos_df.iloc[2:,:].reset_index(drop=True)


oos_df = oos_df.drop(columns=oos_df.columns[[1,9,10,11]])
print(oos_df)


   state  White  Black Hispanic  Asian American Indian or Alaska Native  \
0     AL   0.33  0.611    0.036    <1%                              <1%   
1     AK  0.312  0.015    0.011      0                            0.575   
2     AZ    0.4  0.117    0.375    <1%                            0.027   
3     AR  0.389   0.34     0.17  0.012                              <1%   
4     CA   0.43  0.053      0.4  0.016                            0.041   
5     CO  0.488   0.05    0.391    <1%                              <1%   
6     CT   0.37  0.249    0.287  0.024                                0   
7     DE  0.426  0.459    0.082      0                                0   
8     DC      0      0        0      0                                0   
9     FL  0.395  0.339    0.194    <1%                              <1%   
10    GA  0.356  0.492    0.094    <1%                              <1%   
11    HI    NaN    NaN      NaN    NaN                              NaN   
12    ID  0.673  0.023   

In [44]:
# GRAPHS out of school suspension |bar vertical|
g_width,g_height = get_sizing('6.3')

filename = '6.3.png'
axis_font_size = 38

us_only = oos_df[oos_df['state']=="US"].reset_index(drop=True)

for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    result = oos_df[oos_df['state']==state].reset_index(drop = True)
    print(bordered(state))
    # print(list(result.columns))
    print(result.to_string())
    # print(us_only.to_string())
    ymax,fig = graph_oos(result, us_only, state, g_width,g_height)


    fig.update_layout(
        width=g_width,
        height=g_height,
        xaxis=dict(tickfont=get_base_text(axis_font_size)),
        margin = dict(l=20,t=30,r=20,b=30)
                   
    )
    fig.update_yaxes(
        visible=False, 
        range = [0,ymax+.25]

    )
    if settings.show_it:
        
        fig.show()
        
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)
    if settings.just_one:
        break


# oos_df.columns = ['state', 'year']+list(oos_df.iloc[1,2:])
# oos_df = oos_df.iloc[2:, :].reset_index(drop=True)
# oos_df = oos_df.drop(columns=['notes', 'date pulled'])
# print(oos_df.columns)


┌──┐
│FL│
└──┘
  state  White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    FL  0.395  0.339    0.194   <1%                              <1%                                       <1%             0.064
og ymax: 0.461
ymax: 0.561


┌──┐
│UT│
└──┘
  state  White Black Hispanic  Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    UT  0.533  0.05    0.328  0.015                            0.022                                     0.016             0.036
og ymax: 0.533
ymax: 0.633


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│WI│
└──┘
  state  White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    WI  0.648  0.113    0.102   <1%                            0.027                                       <1%             0.102
og ymax: 0.648
ymax: 0.748


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 7

### 7.1 Public HS Graduation Rate

In [45]:
grad_rate_df = data_pull.get_collected_data(metric='hs_grad_rate')

grad_rate_df.columns = grad_rate_df.iloc[0,:].reset_index(drop=True)
grad_rate_df = grad_rate_df.iloc[2:,:].reset_index(drop=True)
grad_rate_df = grad_rate_df.loc[:,['state', '2022-2023 data', '2023-2024 HS grad rate']]
grad_rate_df.columns = ['state_abrv', '2023', '2024']
grad_rate_df = grad_rate_df.dropna(how='all')


# Melt the DataFrame to long format
grad_rate_df = grad_rate_df.melt(id_vars=['state_abrv'], 
                    var_name='year', 
                    value_name='grad_rate')

# Convert 'year' to int and 'grad_rate' to float
grad_rate_df['year'] = pd.to_numeric(grad_rate_df['year'], errors='coerce')
grad_rate_df['grad_rate'] = pd.to_numeric(grad_rate_df['grad_rate'], errors='coerce')
grad_rate_df = grad_rate_df.sort_values(by=['state_abrv','year']).reset_index(drop=True)
# print(grad_rate_df.to_string())


ref = data_pull.get_collected_data(metric='hs_grad_rate (ref)')
ref = ref.iloc[:,:14]
ref.columns = ['state', 'state_abrv']+ [x.split('-')[-1].strip() for x in ref.columns[2:]]
# print(ref.to_string())

# Assuming you already have the original wide-format DataFrame called `ref`
# First, replace '---' strings with NaN and ensure all values from 2011-2023 are numeric
ref.replace('---', pd.NA, inplace=True)

# Convert all year columns to numeric (safe conversion)
for year in ref.columns[2:]:
    ref[year] = pd.to_numeric(ref[year], errors='coerce')

# Now melt the DataFrame from wide to long format
ref_long = ref.melt(id_vars=['state', 'state_abrv'], 
                  var_name='year', 
                  value_name='grad_rate')

# Optional: Convert 'year' column to int (if all values are valid years)
ref_long['year'] = pd.to_numeric(ref_long['year'], errors='coerce')

# Sort and reset index for cleaner output
ref_long = ref_long.loc[:,["state_abrv", "year", "grad_rate"]]
ref_long = ref_long.sort_values(by=['state_abrv', 'year']).reset_index(drop=True)

print(ref_long.to_string())


    state_abrv  year  grad_rate
0           AK  2011      68.00
1           AK  2012      70.00
2           AK  2013      71.80
3           AK  2014      71.10
4           AK  2015      75.60
5           AK  2016      76.10
6           AK  2017      78.20
7           AK  2018      78.50
8           AK  2019      80.40
9           AK  2020      79.00
10          AK  2021      78.00
11          AK  2022      78.00
12          AL  2011      72.00
13          AL  2012      75.00
14          AL  2013      80.00
15          AL  2014      86.30
16          AL  2015      89.30
17          AL  2016      87.10
18          AL  2017      89.30
19          AL  2018      90.00
20          AL  2019      91.70
21          AL  2020      91.00
22          AL  2021      91.00
23          AL  2022      91.00
24          AR  2011      81.00
25          AR  2012      84.00
26          AR  2013      84.90
27          AR  2014      86.90
28          AR  2015      84.90
29          AR  2016      87.00
30      

In [46]:
full_grad_rate_df = pd.concat([grad_rate_df,ref_long]).sort_values(by=['state_abrv','year']).reset_index(drop=True)
# print(full_grad_rate_df.to_string())

In [47]:
def col_renamer(x):
    clean_x = str(x).lower()
    if 'year' in clean_x:
        return 'year'
    elif 'us' in clean_x:
        return 'us'
    else:
        return 'state'

In [48]:
# GRAPHS overall grad rate |scatter plot|
g_width,g_height = get_sizing('7.1')
import math
us_only = full_grad_rate_df[full_grad_rate_df['state_abrv']=='US'].reset_index(drop=True)
offset = 3
axis_font_size = 45
special = ['ID', 'GA']
filename = f'7.1.png'
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    if state == 'US':
        continue

    result = full_grad_rate_df[full_grad_rate_df['state_abrv']==state].reset_index(drop=True)
    print(bordered(state))

    # print(result.to_string())
    # PRINT(US.ONLY)
    all_values  =result['grad_rate'].to_list()+us_only['grad_rate'].to_list()
    all_values = [x for x in all_values if math.isnan(x)==False]
    y_max = max(all_values)
    y_min = min(all_values)
    # print(y_max)
    # print(y_min)
    y_range_actual = y_max-y_min
    # print(f'actual y range: {y_range_actual}')
    # if y_range_actual<15:

    #     continue
    
    dfs_dict = {'state':result,'us':us_only}
    # for k,v in dfs_dict.items():
    #     new = v.drop(labels = ['jurisdiction', 'grade','subject'], axis = 1)
    #     dfs_dict[k] = new
    
    # print(dfs_dict.get('us').to_string())
    # print(dfs_dict.get('state').to_string())
    merged = pd.concat(dfs_dict.values())
    # # merged = pd.merge(dfs_dict.get('state'),dfs_dict.get('us'),on='year',how = 'outer', suffixes=('_state', '_us'))
    merged = merged.pivot(index = 'year', columns = 'state_abrv', values = 'grad_rate').reset_index()
    # merged.columns = ['year','us','state' ]
    # # df['col_3'] = df.apply(lambda row: f(row['col_1'], row['col_2']), axis=1)
    merged.columns = [col_renamer(x) for x in merged.columns]
    if y_range_actual<15:
        print('Under 15')
        multiplier = 1
        offset=4
    else:
        print('Over 15')
        multiplier = 1.3
        offset=7

    merged['state_position'] = merged.apply(lambda row: position_calc(row['state'],row['us'],offset=offset, multiplier=multiplier), axis=1)
    merged['us_position'] = merged.apply(lambda row: position_calc(row['us'],row['state'],offset=offset, multiplier=multiplier), axis=1)
    
    # print(merged.to_string(max_colwidth=30))

    # too_low = merged[(merged['state']<17)|(merged['us']<17)]
    years = merged['year'].to_list()
    

    position_erros = merged[merged['state_position'].isna()]
    err_indexes = [min(list(position_erros.index))-1]+list(position_erros.index)
    # print(err_indexes)
    position_erros = merged.iloc[err_indexes,:]
    # print(position_erros.to_string(max_colwidth=30))
    # print(merged.to_string(max_colwidth=30))

    # fix nan in state_position
    offset = -4
    for row in position_erros.itertuples(index=True):
        
        if isinstance(row.state_position, float) and 'nan'  in str(row.state_position).lower():
            # print(row)
            
            new = row.state+offset
            merged.loc[row.Index, 'state_position'] = new
            # continue
        else:
            offset = row.state_position-row.state
    merged['us_pos_dif'] = pd.to_numeric(merged['us_position'], errors='coerce')
    
    # fix nan in us_position
    offset = -4

    for row in position_erros.itertuples(index=True):
        
        if isinstance(row.state_position, float) and 'nan'  in str(row.state_position).lower():
            # print(row)
            
            new = row.state+offset
            merged.loc[row.Index, 'state_position'] = new
            # continue
        # else:
        #     offset = row.state_position-row.state

        # print(f'offset: {offset}')
    # print(merged.to_string(max_colwidth=30))
    prev = None

    prev_offset = merged[merged['year']==2022]
    # print(prev_offset)
    prev_offset = prev_offset['state_position'].to_list()[0] - prev_offset['state'].to_list()[0]
    print(prev_offset)
    for row in merged.itertuples(index=True):
        # if row.year >2022:
        #     print(row)
        #     print(type(row.year))
            
        if prev is not None:
            # redo offset if they are the same value
            # print(row)
            if row.state-row.us==0:
                print('both equal')
                offset_state = getattr(prev,'state_position')-getattr(prev,'state')
                merged.loc[row.Index, 'state_position'] = row.state+offset_state
                
                offset_us = getattr(prev,'us_position')-getattr(prev,'us')
                merged.loc[row.Index, 'us_position'] = row.us+offset_us
            elif row.year > 2022:
                    if prev_offset>0:
                        state_diff = row.state_position-row.state
                        new_state_w_offset = row.state+abs(state_diff)
                    elif prev_offset<0:
                        state_diff = row.state_position-row.state
                        new_state_w_offset = row.state-abs(state_diff)
                    

                    # print(f'new_offset: {new_state_w_offset}')
                    merged.loc[row.Index, 'state_position'] = new_state_w_offset
        # print(type(row.year))
        prev = row
    # print('Final Merged')
    # print(merged.to_string(max_colwidth=30))
    


    for row in merged.itertuples(index=True):
        if row.year < 2023:
            continue
        if row.Index == len(merged)-1:
            merged.loc[row.Index,'state_position'] = row.state_position-(row.state*.10)
            break
        diff = merged.at[row.Index+1,'state']-row.state

        perc_o_yrange = abs(diff/y_range_actual)

        if diff < -5:
            merged.loc[row.Index,'state_position'] = row.state_position-(row.state*.10)




    # # if too_low.empty:
    #     y_min = 10
    # else:
    #     y_min = 0


    # y_min  = (min(all_values)-10)
    # print(f'y min/max = {y_min},{y_max}')
    
    fig = graph_grad_rate(merged, g_width, g_height)
    if y_range_actual>30:
        range_space = 15
    else:
        range_space = 10
    y_max_for_range = y_max + range_space 
    y_min_for_range = y_min - range_space
    fig.update_layout(
        width=g_width,
        height=g_height,
        xaxis = dict(tickfont=get_base_text(axis_font_size)),
        yaxis = dict(
            visible = False,
            range = [y_min_for_range, y_max_for_range]
        )
    )

    if settings.show_it:
        fig.show()
        if settings.just_one:
            break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)

    # save_to_folder(fig,filename,g_width,g_height,state)
    # fig.show()
    # break
    


┌──┐
│FL│
└──┘
Over 15
7.0
both equal


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│UT│
└──┘
Under 15
4.0
both equal


┌──┐
│WI│
└──┘
Under 15
3.0


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 7.2 Graduation Rate by Race and Ethnicity

In [49]:
# GRAPHS Grad Rate By Race and Ethnicity |bar vertical|

g_width,g_height = get_sizing('7.2')

filename = f'7.2.png'
grad_rate_by_re = data_pull.get_collected_data(metric='hs_grad_rate_subgroups')
asian_only_states = grad_rate_by_re[grad_rate_by_re['asian only'] == True]['state_abrv'].to_list()
# print(asian_only_states)
# print(grad_rate_by_re.to_string(max_colwidth=20))
axis_font_size = 45


# grad_rate_by_re.columns = ['state_abrv','priority','total', including]

import pandas as pd
# Assuming your DataFrame is called df3
# Example cleanup before melting
grad_rate_by_re = grad_rate_by_re.dropna(subset=['state_abrv', 'updated year'])  # Drop rows missing essential data
# print(grad_rate_by_re.to_string(max_colwidth=20))

# List of columns that contain graduation rate values by group
group_columns = [
    'Total',
    'American Indian / Alaska Native',
    'Asian/Pacific Islander',
    'Hispanic',
    'Black',
    'White',
    'Two or more races'
]

# Melt to long format
grad_rate_by_re = grad_rate_by_re.melt(
    id_vars=['state_abrv', 'updated year'],
    value_vars=group_columns,
    var_name='group',
    value_name='grad_rate'
)
grad_rate_by_re['group'] = grad_rate_by_re['group'].replace({
    'Two or more races': 'Two or More'
})

# Rename for consistency
grad_rate_by_re.rename(columns={'updated year': 'year'}, inplace=True)

# # Drop rows without graduation rate values
# grad_rate_by_re = grad_rate_by_re.dropna(subset=['grad_rate'])

# Optional: convert year to int and grad_rate to float
grad_rate_by_re['year'] = pd.to_numeric(grad_rate_by_re['year'], errors='coerce').astype('Int64')
grad_rate_by_re['grad_rate'] = pd.to_numeric(grad_rate_by_re['grad_rate'], errors='coerce')

# Reset index
grad_rate_by_re = grad_rate_by_re[grad_rate_by_re['state_abrv']!='x'].reset_index(drop=True)

# Preview result
# print(grad_rate_by_re.to_string())

us_only = grad_rate_by_re[grad_rate_by_re['state_abrv']=='US'].reset_index(drop=True)
# print(us_only.to_string())
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    if state == 'US':
        continue
    result = grad_rate_by_re[grad_rate_by_re['state_abrv']==state].reset_index(drop=True)
    print(result.to_string())
    fig = graph_gradrate_re(result, us_only, state, asian_only_states)
    fig.update_layout(
        width=g_width,
        height=g_height
        )
    fig.update_xaxes(
           tickfont=get_base_text(axis_font_size)
        )
    

    if settings.show_it:
        fig.show()
    if settings.just_one:
        break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)





  state_abrv  year                            group  grad_rate
0         FL  2024                            Total  89.700000
1         FL  2024  American Indian / Alaska Native  87.000000
2         FL  2024           Asian/Pacific Islander  96.877673
3         FL  2024                         Hispanic  89.000000
4         FL  2024                            Black  85.300000
5         FL  2024                            White  92.400000
6         FL  2024                      Two or More  89.700000
Original df groups: ['Total' 'American Indian / Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
Original us_df groups: ['Total' 'American Indian / Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'H

  state_abrv  year                            group  grad_rate
0         UT  2024                            Total      0.883
1         UT  2024  American Indian / Alaska Native      0.770
2         UT  2024           Asian/Pacific Islander        NaN
3         UT  2024                         Hispanic      0.810
4         UT  2024                            Black      0.810
5         UT  2024                            White      0.910
6         UT  2024                      Two or More        NaN
Original df groups: ['Total' 'American Indian / Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
Original us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'His

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv  year                            group  grad_rate
0         WI  2024                            Total   0.911000
1         WI  2024  American Indian / Alaska Native   0.853000
2         WI  2024           Asian/Pacific Islander   0.934125
3         WI  2024                         Hispanic   0.851000
4         WI  2024                            Black   0.717000
5         WI  2024                            White   0.951000
6         WI  2024                      Two or More   0.870000
Original df groups: ['Total' 'American Indian / Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
Original us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'His

C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 7.3 Graduation Rate by Other subgroup

In [50]:
# GRAPHS Grad Rate By Other Subgroups |bar vertical|
g_width,g_height = get_sizing('7.3')

axis_font_size = 55

filename = f'7.3.png'

grad_rate_by_re = data_pull.get_collected_data(metric='hs_grad_rate_subgroups')

# grad_rate_by_re.columns = ['state_abrv','priority','total', including]

import pandas as pd

# Assuming your DataFrame is called df3
# Example cleanup before melting
grad_rate_by_re = grad_rate_by_re.dropna(subset=['state_abrv', 'updated year'])  # Drop rows missing essential data

# List of columns that contain graduation rate values by group
group_columns = [
    'Economically disadvantaged (Based on state definition)',
    'Limited English proficiency',
    'Students with disabilities'
]


# Melt to long format
grad_rate_by_re = grad_rate_by_re.melt(
    id_vars=['state_abrv', 'updated year'],
    value_vars=group_columns,
    var_name='group',
    value_name='grad_rate'
)
grad_rate_by_re['group'] = grad_rate_by_re['group'].replace({
    'Two or more races': 'Two or More'
})


# print(grad_rate_by_re.to_string())
# for row in grad_rate_by_re.itertuples():
#     if 

# Rename for consistency
grad_rate_by_re.rename(columns={'updated year': 'year'}, inplace=True)

# # Drop rows without graduation rate values
# grad_rate_by_re = grad_rate_by_re.dropna(subset=['grad_rate'])

# Optional: convert year to int and grad_rate to float
grad_rate_by_re['year'] = pd.to_numeric(grad_rate_by_re['year'], errors='coerce').astype('Int64')
grad_rate_by_re['grad_rate'] = pd.to_numeric(grad_rate_by_re['grad_rate'], errors='coerce')

# Reset index
grad_rate_by_re = grad_rate_by_re[grad_rate_by_re['state_abrv']!='x'].reset_index(drop=True)

# Preview result
# print(grad_rate_by_re.to_string())

us_only = grad_rate_by_re[grad_rate_by_re['state_abrv']=='US'].reset_index(drop=True)
# print(us_only.to_string())
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
                continue
    if state == 'US':
        continue
    result = grad_rate_by_re[grad_rate_by_re['state_abrv']==state].reset_index(drop=True)
    print(result.to_string())
    fig = graph_gradrate_other_subgroup(group_columns,result, us_only, state)
    
    fig.update_layout(
        width=g_width,
        height=g_height,
        yaxis=dict(tickfont=get_base_text(axis_font_size))
    )
    
    fig.update_yaxes(
           visible=False
        )

    if settings.show_it:
        fig.show()
        if settings.just_one:
            break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)







  state_abrv  year                                                   group  grad_rate
0         FL  2024  Economically disadvantaged (Based on state definition)       86.2
1         FL  2024                             Limited English proficiency       80.7
2         FL  2024                              Students with disabilities       86.8


  state_abrv  year                                                   group  grad_rate
0         UT  2024  Economically disadvantaged (Based on state definition)       0.79
1         UT  2024                             Limited English proficiency       0.77
2         UT  2024                              Students with disabilities       0.74


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv  year                                                   group  grad_rate
0         WI  2024  Economically disadvantaged (Based on state definition)      0.839
1         WI  2024                             Limited English proficiency      0.787
2         WI  2024                              Students with disabilities      0.748


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 8

### 8.1: Dropout Rate

In [51]:
# set up
dropout_df = data_pull.get_collected_data(metric = 'dropout_rate')
dropout_df = dropout_df.iloc[:,:11]
# print(dropout_df.to_string(max_colwidth=30))


# Step 1: Rename columns properly
dropout_df.columns = [
    'state_abrv', 'priority', 'source', 'updated_year', 'Total',
    'White', 'Black', 'Hispanic', 'Asian/Pacific Islander',
    'American Indian/Alaska Native', 'Two or more races'
]

# Step 2: Keep only rows where state_abrv is not NaN or 'x'
df_clean = dropout_df[~dropout_df['state_abrv'].isna()]
df_clean = df_clean[df_clean['state_abrv'] != 'x']

# Step 3: Select columns of interest
group_columns = [
    'Total', 'White', 'Black', 'Hispanic', 'Asian/Pacific Islander',
    'American Indian/Alaska Native', 'Two or more races'
]
df_clean = df_clean[['state_abrv', 'updated_year'] + group_columns]

# Step 4: Convert updated_year to a single year (pick later if range)
def extract_later_year(val):
    if pd.isna(val):
        return np.nan
    # Match years in format "YYYY" or "YYYY-YYYY"
    years = re.findall(r'\d{4}', str(val))
    if not years:
        return np.nan
    return int(years[-1])  # pick the later year

df_clean['updated_year'] = df_clean['updated_year'].apply(extract_later_year)

# Step 5: Melt to long format
df_long = df_clean.melt(
    id_vars=['state_abrv', 'updated_year'],
    value_vars=group_columns,
    var_name='group',
    value_name='value'
)

print(df_long.sort_values(by='state_abrv', ascending=True).to_string(max_colwidth=20))
format_dict = {}
for state in state_abbreviations:
    # print(bordered(state))
    result = df_long[df_long['state_abrv']==state]
    values = [x for x in result['value'].to_list() if 'nan' not in str(x).lower()]
    max_val = max(values)
    min_val = min(values)
    # print(max_val)
    if max_val<1:
        # print('its a decimal')
        format_dict[state] = 'decimal'
    elif max_val>1:
        format_dict[state] = 'whole'
        # print('its a whole')

print(format_dict)
# Step 6: Convert values to float and standardize percentages
def normalize_percentage(x, state_abrv):
    # print(state_abrv)
    if str(x).lower()=='nan':
        return x
    
    state_val = format_dict.get(state_abrv)
    if state_val == 'decimal':
        return x
    elif state_val == 'whole':
        return x/100
    
    # try:
    #     x = float(x)
    #     if x > 1.5:  # Likely stored as 35.1 → treat as 35.1%
    #         return x
    #     else:        # Already decimal (0.351 → 35.1%)
    #         return x * 100
    # except:
    #     return np.nan
# print(df_long.columns)

# df['col_3'] = df.apply(lambda x: get_sublist(x.col_1, x.col_2), axis=1)
df_long['value'] = df_long.apply(lambda x: normalize_percentage(x.value, x.state_abrv), axis=1)

# Step 7: Rename column
df_long.rename(columns={'updated_year': 'year'}, inplace=True)

# Optional: sort for readability
dropout_df = df_long.sort_values(['state_abrv', 'group']).reset_index(drop=True)
print(dropout_df.to_string(max_colwidth=30))



    state_abrv  updated_year                group                value
320         AK        2024.0    Two or more races                0.038
2           AK        2024.0                Total               0.0356
214         AK        2024.0  Asian/Pacific Is...                 0.03
161         AK        2024.0             Hispanic                0.032
55          AK        2024.0                White                0.023
108         AK        2024.0                Black                0.035
267         AK        2024.0  American Indian/...                0.065
1           AL        2023.0                Total                0.017
213         AL        2023.0  Asian/Pacific Is...                0.051
160         AL        2023.0             Hispanic                0.026
54          AL        2023.0                White                0.012
107         AL        2023.0                Black                0.023
266         AL        2023.0  American Indian/...                0.015
319   

In [52]:
# GRAPHS Dropout rate by Race and Ethnicity
g_width,g_height = get_sizing('8.1')
axis_font_size = 45

filename = f'8.1.png'
# Reset index

# Preview result
# print(grad_rate_by_re.to_string())

us_only = dropout_df[dropout_df['state_abrv']=='US'].reset_index(drop=True)
# print(us_only.to_string())
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    print(state)
    if state == 'US':

        continue
    result = dropout_df[dropout_df['state_abrv']==state].reset_index(drop=True)
    # print(result.to_string())
    fig = graph_dropouts(result, us_only, state, debug=False)
    fig.update_layout(
        width=g_width,
        height=g_height,
        xaxis=dict(tickfont=get_base_text(axis_font_size)))
    
    fig.update_yaxes(
           visible=False)
    if settings.show_it:
        fig.show()
    if settings.just_one:
        break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)


FL
[0.022, 0.044, 0.0055312232677502135, 0.023, 0.027, 0.02, 0.025]


UT
[0.043, 0.076, 0.14, 0.07, 0.078, 0.033, 0.044]


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




WI
[0.018, 0.039, 0.024, 0.031, 0.062, 0.009, 0.023]


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 8.2: CTE concentratos

In [53]:
settings.subset_states

['FL', 'UT', 'WI']

In [54]:
# GRAPHS
cte_df = data_pull.cte_calculated()
# print(cte_df.to_string())
of_percent = cte_df['perc_o_cte'].to_list()
of_enrollment = cte_df['perc_o_enr'].to_list()
all_values = of_percent+of_enrollment
# print(max([x for x in all_values if 'n/a' not in str(x).lower()]))
axis_font_size = 45

# GRAPHS Grad Rate By Race and Ethnicity
g_width,g_height = get_sizing('8.2')

filename = f'8.2.png'
grad_rate_by_re = data_pull.get_collected_data(metric='hs_grad_rate_subgroups')
print(settings.subset_states)
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            print(f'skipping {state}')
            continue
    if state == 'US':
        continue
    fig = graph_8_cte(cte_df,state)
    fig.update_yaxes(
        visible=False
    )
    fig.update_layout(
    width=g_width,
    height=g_height,
        xaxis=dict(tickfont=get_base_text(axis_font_size))
    )
    if settings.show_it:
        fig.show()
    if settings.just_one:
        break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)



['FL', 'UT', 'WI']
skipping AZ
skipping CA
skipping CT
skipping DC
skipping DE
  state                   group perc_o_cte  perc_o_enr  order
0    FL            nat_am_or_ak   0.002377    0.002358      1
5    FL  asian_pacific_islander   0.032502    0.029987      2
2    FL                hispanic    0.35869    0.363734      3
1    FL                   black   0.205894    0.209480      4
3    FL                   white   0.367046    0.353307      5
4    FL             two_or_more   0.033489    0.041134      6


skipping GA
skipping IA
skipping ID
skipping IL
skipping IN
skipping KS
skipping KY
skipping MA
skipping MD
skipping MI
skipping MN
skipping MO
skipping NC
skipping ND
skipping NE
skipping NJ
skipping NM
skipping NY
skipping OH
skipping OK
skipping OR
skipping RI
skipping SC
skipping TN
skipping TX
  state                   group perc_o_cte  perc_o_enr  order
0    UT            nat_am_or_ak   0.009671    0.009591      1
5    UT  asian_pacific_islander     0.0317    0.033487      2
2    UT                hispanic   0.185704    0.196381      3
1    UT                   black   0.013092    0.013315      4
3    UT                   white   0.729017    0.712951      5
4    UT             two_or_more   0.030816    0.034155      6


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




skipping VA
skipping VT
skipping WA
  state                   group perc_o_cte  perc_o_enr  order
0    WI            nat_am_or_ak   0.008376    0.010300      1
5    WI  asian_pacific_islander   0.037398    0.042946      2
2    WI                hispanic    0.11436    0.135865      3
1    WI                   black   0.057021    0.087370      4
3    WI                   white   0.744047    0.671659      5
4    WI             two_or_more   0.038798    0.051431      6


C:\Users\clutz\AppData\Local\Temp\ipykernel_33952\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




skipping WV
skipping WY


### AP graphs 


#### 8.3: AP Participation

In [11]:
# GRAPHS participation

sat_df = data_pull.get_ap_graph_data()
# print(sat_df.to_string())
g_width,g_height = get_sizing('8.3')
axis_font_size = 45

filename = f'8.3.png'

for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    if state =="US":
        continue
    us_only = sat_df[sat_df['state_abrv']=="US"]['perc_in_an_ap'].to_list()
    state_only = sat_df[sat_df['state_abrv']==state]['perc_in_an_ap'].to_list()

    print(us_only)
    print(state_only)
    
    fig = graph_ap(us_only[0],state_only[0],state, 'percent')
    fig.update_layout(
        
        width=g_width,
        height=g_height,
        margin = dict(l=0,t=10,b=0,r=0),

        xaxis=dict(tickfont=get_base_text(axis_font_size),
                   title = dict(
                       text = wrap_category_name('% of 11th Graders Enrolled In At Least One AP Course', 30),
                       font = get_base_text(40),
                    standoff=60)))
    
    fig.update_yaxes(
           visible=False
        )
    
    if settings.show_it:
        fig.show()
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)
    if settings.just_one:
        break


    # break


[0.3532730189747818]
[0.41582996257513327]


C:\Users\clutz\AppData\Local\Temp\ipykernel_10960\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




[0.3532730189747818]
[0.30125073715155837]


[0.3532730189747818]
[0.35333628529582073]


#### 8.4: AP exams per 1000

In [12]:
# GRAPHS num ap exams per 1000
sat_df = data_pull.get_ap_graph_data()
# print(sat_df.to_string())
g_width,g_height = get_sizing('8.4')
filename = f'8.4.png'
axis_font_size = 45


for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    if state =="US":
        continue
    us_only = sat_df[sat_df['state_abrv']=="US"]['ap_exams_per_k_11_12'].to_list()
    state_only = sat_df[sat_df['state_abrv']==state]['ap_exams_per_k_11_12'].to_list()

    print(us_only)
    print(state_only)
    
    fig = graph_ap(us_only[0],state_only[0],state, 'number')
    fig.update_layout(
        width=g_width,
        height=g_height,
        margin = dict(l=0,t=10,b=0,r=0),
        
        xaxis=dict(tickfont=get_base_text(axis_font_size),
                   title = dict(
                       text = wrap_category_name('# of AP Exams Per 1,000 11th-12th Graders', 25),
                       font = get_base_text(40),
                    standoff=60)))
    
    fig.update_yaxes(
           visible=False
        )

    if settings.show_it:
        fig.show()
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)
    if settings.just_one:
        break

    # break


[500.371745167851]
[549.402714247911]


[500.371745167851]
[288.153644389286]


C:\Users\clutz\AppData\Local\Temp\ipykernel_10960\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




[500.371745167851]
[477.924997445417]


#### 8.5: AP % Passing

In [13]:
#GRAPHS Percent passing
sat_df = data_pull.get_ap_graph_data()
# print(sat_df.to_string())
g_width,g_height = get_sizing('8.5')
filename = f'8.5.png'
axis_font_size = 45


for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    if state =="US":
        continue
    us_only = sat_df[sat_df['state_abrv']=="US"]['perc_3_orbetter'].to_list()
    state_only = sat_df[sat_df['state_abrv']==state]['perc_3_orbetter'].to_list()

    print(us_only)
    print(state_only)
    
    fig = graph_ap(us_only[0],state_only[0],state, 'percent')
    
    fig.update_layout(
        width=g_width,
        height=g_height,
        margin = dict(l=0,t=10,b=0,r=0),

        xaxis=dict(tickfont=get_base_text(axis_font_size),
            title = dict(
                text = wrap_category_name('% of AP Exams With Passing Grades', 18),
                font = get_base_text(40),
            standoff=60)))

    fig.update_yaxes(
           visible=False
        )
    

    if settings.show_it:
        fig.show()
    if settings.save_it:
        print(g_width,g_height)
        save_to_folder(fig,filename,g_width,g_height,state)
    if settings.just_one:
        break


[0.660227674346602]
[0.645210412396805]


825 470
[0.660227674346602]
[0.726925092687851]


C:\Users\clutz\AppData\Local\Temp\ipykernel_10960\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




825 470
[0.660227674346602]
[0.720719904918218]


825 470


## Page 9

### 9.1 College Entrance Exams

In [14]:
# College Entrance Exams

g_width,g_height = get_sizing('9.1')

filename = f'9.1.png'
axis_font_size = 45

#data set up
act_df = data_pull.get_collected_data(metric='act_benchmarks', no_header=True)
act_df = act_df.iloc[1:,1:].reset_index(drop = True)
act_df.columns = act_df.iloc[0,:]

act_df = act_df.iloc[1:,:].reset_index(drop=True)
# print(act_df.to_string())


us_only = act_df[act_df['State']=='US']
print(us_only.to_string())
for state in state_abbreviations_priority:
    if settings.subset_states != 'None' and state not in settings.subset_states:
            continue
    result = act_df[act_df['State']==state]
    # print(result.to_string())
    fig = graph_act_benchmarks(result, us_only, state, g_width, g_height)
    fig.update_yaxes(
        visible=False
    )
    fig.update_layout(
    width=g_width,
    height=g_height,
        xaxis=dict(tickfont=get_base_text(axis_font_size)))

    if settings.show_it:
        fig.show()
        if settings.just_one:
            break
    if settings.save_it:
        save_to_folder(fig,filename,g_width,g_height,state)
    



0 State Total Black American Indian/Alaska Native White Hispanic Asian Native Hawaiian/Other PI Two or more races
0    US   0.3  0.09                           0.1  0.39     0.17   0.6                     0.11              0.31
['Total', 'Black', 'American Indian/Alaska Native', 'White', 'Hispanic', 'Asian', 'Native Hawaiian/Other PI', 'Two or more races']
['30%', '9%', '10%', '39%', '17%', '60%', '11%', '31%']


['Total', 'Black', 'American Indian/Alaska Native', 'White', 'Hispanic', 'Asian', 'Native Hawaiian/Other PI', 'Two or more races']
['30%', '9%', '10%', '39%', '17%', '60%', '11%', '31%']


C:\Users\clutz\AppData\Local\Temp\ipykernel_10960\3341793293.py:12: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['Total', 'Black', 'American Indian/Alaska Native', 'White', 'Hispanic', 'Asian', 'Native Hawaiian/Other PI', 'Two or more races']
['30%', '9%', '10%', '39%', '17%', '60%', '11%', '31%']
